# Статистический анализ всех полей Сферы и ЕГРН

Ноутбук собирает расширенный датасет объектов недвижимости и считает статистику по всем доступным полям Сферы и ЕГРН.

Для соединения используется второй вариант:

1. `full_address` из Сферы передаётся в CDI без изменений;
2. из CDI берётся один ФИАС дома;
3. в ЕГРН по этому ФИАС ищутся только записи с `OKS_TYPE = здание`;
4. данные ЕГРН присоединяются только тогда, когда найдено ровно одно здание.

Площадь при соединении не используется. Если найдено несколько зданий, данные ЕГРН остаются пустыми, а строка отмечается как неоднозначная.

Промежуточный CSV с объектами не создаётся. Сохраняется один агрегированный паспорт признаков. Реальные адреса, ИНН, названия компаний, номера договоров и кадастровые номера в частотные значения паспорта не выводятся.


In [ ]:
%pip install pandas numpy sqlalchemy "psycopg[binary]" oracledb openpyxl


In [ ]:
import json
from pathlib import Path
import oracledb
import numpy as np
import pandas as pd
import re
from sqlalchemy import URL, create_engine, text

pd.set_option('display.max_columns', 100)

In [ ]:
CURRENT_DIR = Path.cwd()
if (CURRENT_DIR / 'уч данные.txt').exists():
    NOTEBOOK_DIR = CURRENT_DIR
elif (CURRENT_DIR / 'notebooks' / 'уч данные.txt').exists():
    NOTEBOOK_DIR = CURRENT_DIR / 'notebooks'
else:
    NOTEBOOK_DIR = CURRENT_DIR

PROJECT_ROOT = (
    NOTEBOOK_DIR.parent
    if NOTEBOOK_DIR.name == 'notebooks'
    else NOTEBOOK_DIR
)
OUTPUT_DIR = PROJECT_ROOT / 'РЕЗУЛЬТАТЫ_ЛОКАЛЬНО'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Корень проекта:', PROJECT_ROOT)
print('Папка результатов:', OUTPUT_DIR)

# 1. Подключение к Сфере


In [ ]:
CREDENTIALS_PATH = NOTEBOOK_DIR / 'уч данные.txt'

def read_credentials(path):
    if not path.exists():
        raise FileNotFoundError(f'Не найден файл с учётными данными: {path}')

    credentials = {}
    for line_number, raw_line in enumerate(
        path.read_text(encoding='utf-8-sig').splitlines(),
        start=1,
    ):
        line = raw_line.strip()
        if not line or line.startswith('#'):
            continue
        if '=' not in line:
            raise ValueError(
                f'Строка {line_number}: ожидается запись КЛЮЧ=значение'
            )

        key, value = line.split('=', 1)
        credentials[key.strip()] = value.strip()

    return credentials

credentials = read_credentials(CREDENTIALS_PATH)

sphere_required = [
    'SPHERE_HOST',
    'SPHERE_DATABASE',
    'SPHERE_USER',
    'SPHERE_PASSWORD',
]
sphere_missing = [key for key in sphere_required if not credentials.get(key)]
if sphere_missing:
    raise ValueError(
        'Заполни в уч данные.txt: ' + ', '.join(sphere_missing)
    )

SPHERE_HOST = credentials['SPHERE_HOST']
SPHERE_PORT = int(credentials.get('SPHERE_PORT', '5432'))
SPHERE_DATABASE = credentials['SPHERE_DATABASE']
SPHERE_USER = credentials['SPHERE_USER']
SPHERE_PASSWORD = credentials['SPHERE_PASSWORD']

connection_url = URL.create(
    drivername='postgresql+psycopg',
    username=SPHERE_USER,
    password=SPHERE_PASSWORD,
    host=SPHERE_HOST,
    port=SPHERE_PORT,
    database=SPHERE_DATABASE,
)
engine = create_engine(connection_url, pool_pre_ping=True)

print('Учётные данные прочитаны, подключение к Сфере создано')

In [ ]:
with engine.connect() as connection:
    connection_check = pd.read_sql_query(
        text('select current_database() as database_name, current_user as user_name'),
        connection,
    )

display(connection_check)

# 2. Подключение к Oracle КХД



In [ ]:
khd_required = [
    'KHD_HOST',
    'KHD_SERVICE_NAME',
    'KHD_USER',
    'KHD_PASSWORD',
]
khd_missing = [key for key in khd_required if not credentials.get(key)]
if khd_missing:
    raise ValueError(
        'Заполни в уч данные.txt: ' + ', '.join(khd_missing)
    )

KHD_HOST = credentials['KHD_HOST']
KHD_PORT = int(credentials.get('KHD_PORT', '1521'))
KHD_SERVICE_NAME = credentials['KHD_SERVICE_NAME']
KHD_USER = credentials['KHD_USER']
KHD_PASSWORD = credentials['KHD_PASSWORD']
KHD_DATA_SCHEMA = credentials.get('KHD_DATA_SCHEMA', 'DM_RISK_AVATAR')

khd_dsn = oracledb.makedsn(
    KHD_HOST,
    KHD_PORT,
    service_name=KHD_SERVICE_NAME,
)
khd_connection = oracledb.connect(
    user=KHD_USER,
    password=KHD_PASSWORD,
    dsn=khd_dsn,
)

print('Подключение к КХД создано')


In [ ]:
with khd_connection.cursor() as cursor:
    cursor.execute(
        "select user, sys_context('USERENV', 'DB_NAME') from dual"
    )
    khd_user_name, khd_database_name = cursor.fetchone()

    cursor.execute(
        """
        select table_name
        from all_tables
        where owner = :owner
          and table_name = 'EGRN_DATA'
        order by table_name
        """,
        owner=KHD_DATA_SCHEMA.upper(),
    )
    available_khd_tables = [row[0] for row in cursor.fetchall()]

print('Пользователь КХД:', khd_user_name)
print('База КХД:', khd_database_name)
print('Доступные таблицы:', available_khd_tables)

expected_khd_tables = {'EGRN_DATA'}
missing_khd_tables = sorted(expected_khd_tables - set(available_khd_tables))
if missing_khd_tables:
    raise PermissionError(
        'Не видны таблицы КХД: ' + ', '.join(missing_khd_tables)
    )

with khd_connection.cursor() as cursor:
    cursor.execute(
        """
        select column_name
        from all_tab_columns
        where owner = :owner
          and table_name = 'EGRN_DATA'
        order by column_id
        """,
        owner=KHD_DATA_SCHEMA.upper(),
    )
    egrn_source_columns = [row[0] for row in cursor.fetchall()]

if not egrn_source_columns:
    raise PermissionError('Не удалось получить список колонок EGRN_DATA')

print('Колонок в EGRN_DATA:', len(egrn_source_columns))


# 3. SQL Сфера, расширенный подход


In [ ]:
expanded_sql = r"""
/*
запускать в сфере

запрос собирает все неудаленные объекты недвижимости
если объект связан с подходящим договором данные договора заполняются
если связь не найдена объект остается в результате с пустыми полями договора

одна строка для связанного объекта означает объект в одном договоре
одна строка для несвязанного объекта означает его последнюю версию характеристик
*/

with task_candidates as (
    /* отбираем подходящие задачи оформления */
    select
        t.id as task_id,
        r.id as request_id,
        c.id as contract_id,
        row_number() over (
            partition by c.id
            order by
                coalesce(
                    t.d_conclusion_ins_contract::timestamp with time zone,
                    t.d_create,
                    t.d_change
                ) desc nulls last,
                t.d_create desc nulls last,
                t.d_change desc nulls last,
                t.id desc
        ) as task_number
    from bps_request_ins_task t
    join bps_request_ins r
        on r.id = t.request_ins_id
    join bps_contract c
        on c.id = r.contract_id
    where t.task_type = 'draft_contract'
      and t.status = 'operational_archive'
      and (
          t.ins_document_type = 'new_ins_contract'
          or t.ins_document_type = 'ins_contract_prolong'
          or t.ins_document_type is null
      )
      and t.ins_refuse is not true
      and t.d_delete is null
      and r.d_delete is null
      and c.d_delete is null
),

selected_tasks as (
    /* оставляем последнюю подходящую задачу каждого договора */
    select
        task_id,
        request_id,
        contract_id
    from task_candidates
    where task_number = 1
),

linked_object_candidates as (
    /* находим недвижимость в выбранных задачах */
    select
        ch.insurance_object_id as object_id,
        ch.id as characteristics_id,
        link.id as task_object_link_id,
        selected.task_id,
        selected.request_id,
        selected.contract_id,
        row_number() over (
            partition by selected.task_id, ch.insurance_object_id
            order by
                link.d_change desc nulls last,
                link.d_create desc nulls last,
                ch.version_start_date desc nulls last,
                ch.version_number desc nulls last,
                link.id desc
        ) as link_number
    from selected_tasks selected
    join bps_request_ins_task_insurance_object link
        on link.parent_id = selected.task_id
    join base_insurance_object_characteristics ch
        on ch.id = link.characteristics_id
    join base_insurance_object obj
        on obj.id = ch.insurance_object_id
    where obj.elementary_obj_type = 'nedv_ul_and_ip'
      and obj.d_delete is null
),

selected_links as (
    /* убираем повторные связи одного объекта с одной задачей */
    select
        object_id,
        characteristics_id,
        task_object_link_id,
        task_id,
        request_id,
        contract_id
    from linked_object_candidates
    where link_number = 1
),

object_versions as (
    /* нумеруем версии характеристик каждого объекта */
    select
        obj.id as object_id,
        ch.id as characteristics_id,
        row_number() over (
            partition by obj.id
            order by
                ch.version_is_active desc nulls last,
                ch.version_number desc nulls last,
                ch.version_start_date desc nulls last,
                ch.id desc nulls last
        ) as version_number
    from base_insurance_object obj
    left join base_insurance_object_characteristics ch
        on ch.insurance_object_id = obj.id
    where obj.elementary_obj_type = 'nedv_ul_and_ip'
      and obj.d_delete is null
),

dataset_keys as (
    /* сохраняем все найденные связи с договорами */
    select
        linked.object_id,
        linked.characteristics_id,
        linked.task_object_link_id,
        linked.task_id,
        linked.request_id,
        linked.contract_id,
        'linked'::text as row_source
    from selected_links linked

    union all

    /* добавляем объекты для которых подходящий договор не найден */
    select
        version.object_id,
        version.characteristics_id,
        null::integer as task_object_link_id,
        null::integer as task_id,
        null::integer as request_id,
        null::integer as contract_id,
        'not_linked'::text as row_source
    from object_versions version
    where version.version_number = 1
      and not exists (
          select 1
          from selected_links linked
          where linked.object_id = version.object_id
      )
),

object_link_profile as (
    /* считаем со сколькими договорами связан объект */
    select
        object_id,
        count(distinct contract_id) as contract_count
    from selected_links
    group by object_id
),

selected_characteristics as (
    /* ограничиваем расчет условий версиями из итоговой выборки */
    select distinct characteristics_id
    from dataset_keys
    where characteristics_id is not null
),

condition_summary as (
    /* сворачиваем варианты условий в одну строку */
    select
        cond.characteristics_id,
        count(*) as condition_count,
        count(cond.insured_sum) as filled_insured_sum_count,
        count(distinct cond.insured_sum) filter (
            where cond.insured_sum is not null
        ) as distinct_insured_sum_count,
        min(cond.insured_sum) as minimum_insured_sum,
        max(cond.insured_sum) as maximum_insured_sum,
        count(distinct cond.insured_sum_currency) filter (
            where cond.insured_sum_currency is not null
        ) as currency_count,
        string_agg(
            distinct cond.insured_sum_currency,
            ', '
            order by cond.insured_sum_currency
        ) filter (
            where cond.insured_sum_currency is not null
        ) as insured_sum_currency,
        array_agg(
            distinct cond.terms_option_number
            order by cond.terms_option_number
        ) filter (
            where cond.terms_option_number is not null
        ) as terms_option_numbers,
        min(cond.per_occurance_limit) as minimum_per_occurrence_limit,
        max(cond.per_occurance_limit) as maximum_per_occurrence_limit,
        jsonb_agg(
            to_jsonb(cond)
            order by cond.id
        ) as source_json__base_insurance_object_conditions,
        case
            when count(distinct cond.insured_sum) filter (
                where cond.insured_sum is not null
            ) = 1
             and count(distinct cond.insured_sum_currency) filter (
                where cond.insured_sum_currency is not null
            ) <= 1
            then max(cond.insured_sum)
        end as insured_sum
    from base_insurance_object_conditions cond
    join selected_characteristics selected
        on selected.characteristics_id = cond.characteristics_id
    group by cond.characteristics_id
),

raw_result as (
/* собираем исходные поля расширенного датасета */
select
    /* качество строки */
    keys.row_source,
    case
        when coalesce(profile.contract_count, 0) = 0 then 'not_linked'
        when profile.contract_count = 1 then 'linked'
        else 'multiple_contracts'
    end as contract_link_status,
    coalesce(profile.contract_count, 0) as contract_count,
    (keys.contract_id is not null) as has_contract,
    (address.id is not null) as has_address,
    (conditions.insured_sum is not null) as has_target,
    case
        when conditions.condition_count is null then 'no_conditions'
        when conditions.filled_insured_sum_count = 0 then 'target_is_empty'
        when conditions.distinct_insured_sum_count > 1 then 'several_target_values'
        when conditions.currency_count > 1 then 'several_currencies'
        when conditions.insured_sum <= 0 then 'target_is_not_positive'
        else 'target_is_usable'
    end as target_status,

    /* идентификаторы */
    keys.object_id,
    keys.characteristics_id,
    keys.task_object_link_id,
    keys.task_id,
    keys.request_id,
    keys.contract_id,
    obj.geo_address_id,
    contract.contractor_id as policyholder_id,
    request.corporate_crm_id,

    /* целевая страховая сумма */
    conditions.insured_sum,
    conditions.insured_sum_currency,
    conditions.condition_count,
    conditions.filled_insured_sum_count,
    conditions.distinct_insured_sum_count,
    conditions.minimum_insured_sum as condition_min_insured_sum,
    conditions.maximum_insured_sum as condition_max_insured_sum,
    conditions.currency_count as condition_currency_count,
    conditions.terms_option_numbers,

    /* контрольные суммы */
    task_link.insured_sum as task_object_insured_sum,
    task_link.insured_sum_currency as task_object_insured_sum_currency,
    task.total_ins_contract_amount as contract_insured_sum,
    task.curr_ins_contract_amount as contract_amount_currency,
    task.total_ins_contract_premium as contract_premium,
    ch.insurance_value,
    ch.insurance_value_currency,
    ch.insurance_value_basis,
    ch.pledged_value,
    conditions.minimum_per_occurrence_limit,
    conditions.maximum_per_occurrence_limit,

    /* объект */
    obj.obj_type as object_type,
    obj.elementary_obj_type,
    obj.obj_name as object_name,
    obj.description as object_description,
    obj.original_address,
    obj.active as object_is_active,
    obj.d_create as object_create_date,
    obj.d_change as object_change_date,

    /* характеристики объекта */
    ch.version_number as characteristics_version_number,
    ch.version_start_date as characteristics_version_start_date,
    ch.version_end_date as characteristics_version_end_date,
    ch.version_is_active as characteristics_version_is_active,
    ch.ownership_type,
    ch.is_pledged,
    ch.is_leased,
    ch.insured_components,
    ch.activity_types,
    ch.risk_natures,
    ch.insurance_territory,
    ch.has_losses,
    ch.insurance_object_loss_history,
    ch.characteristics ->> 'total_area_sq_m' as total_area,
    ch.characteristics ->> 'occupied_area_sq_m' as occupied_area,
    ch.characteristics ->> 'construction_year' as construction_year,
    ch.characteristics ->> 'last_capital_repair_year' as capital_repair_year,
    ch.characteristics ->> 'total_floors_count' as floors_count,
    ch.characteristics ->> 'occupied_floor' as occupied_floor,
    ch.characteristics ->> 'load_bearing_walls_material' as walls_material,
    ch.characteristics ->> 'interfloor_overlap_material' as overlap_material,
    ch.characteristics ->> 'roofing_material' as roofing_material,
    ch.characteristics ->> 'fire_alarm_system_availability'
        as fire_alarm_system_availability,
    ch.characteristics ->> 'fire_suppression_system_availability'
        as fire_suppression_system_availability,
    ch.characteristics ->> 'nearest_fire_station_distance_km'
        as nearest_fire_station_distance_km,
    ch.characteristics as object_characteristics_json,

    /* адрес */
    address.full_address,
    address.postal_code,
    address.region_id as address_region_id,
    address.area as district,
    address.settlement_type,
    address.settlement,
    address.street_type,
    address.street,
    address.house,
    address.building,
    address.block,
    address.flat,
    address.office,
    address.fias_code,
    address.longitude,
    address.latitude,
    address.address_dgis_id,

    /* договор */
    contract.n_contract as contract_number,
    contract.document_status as contract_status,
    contract.system_type as contract_source_system,
    contract.ins_product_sbs as insurance_product,
    contract.d_sign_contract as contract_sign_date,
    contract.d_start_contract as contract_start_date,
    contract.d_end_contract as contract_end_date,
    contract.prevcontract_id as previous_contract_id,
    contract.rootcontract_id as root_contract_id,
    previous_contract.n_contract as previous_contract_number,
    previous_contract.d_start_contract as previous_contract_start_date,
    previous_contract.d_end_contract as previous_contract_end_date,

    /* задача и заявка */
    task.task_type,
    task.status as task_status,
    task.ins_document_type,
    task.ins_refuse,
    task.d_create as task_create_date,
    task.d_conclusion_ins_contract as contract_conclusion_date,
    task.ins_product as task_product,
    task.industry as task_industry,
    task.subindustry as task_subindustry,
    task.locations_count,
    task.multi_location,
    task.object_description as task_object_description,
    request.business_segment,
    request.sale_channel,
    request.ins_product as request_product,

    /* страхователь и crm */
    policyholder.inn as policyholder_inn,
    policyholder.company_name_short as policyholder_name,
    policyholder.cdi_id as policyholder_cdi_id,
    crm.segment as crm_segment,
    crm.macroindustry as crm_macroindustry,
    crm.industry as crm_industry,
    crm.okved as crm_okved,

    /* полные строки таблиц для статистического исследования */
    to_jsonb(obj) as source_json__base_insurance_object,
    to_jsonb(ch) as source_json__base_insurance_object_characteristics,
    conditions.source_json__base_insurance_object_conditions,
    to_jsonb(address) as source_json__base_geo_address,
    to_jsonb(task_link) as source_json__bps_request_ins_task_insurance_object,
    to_jsonb(task) as source_json__bps_request_ins_task,
    to_jsonb(request) as source_json__bps_request_ins,
    to_jsonb(contract) as source_json__bps_contract,
    to_jsonb(previous_contract) as source_json__bps_contract_previous,
    to_jsonb(policyholder) as source_json__bps_contractor,
    to_jsonb(crm) as source_json__bps_corporate_crm,

    /* дата состояния строки */
    coalesce(
        task.d_conclusion_ins_contract::timestamp with time zone,
        contract.d_sign_contract,
        ch.version_start_date,
        obj.d_create
    ) as as_of_date
from dataset_keys keys
join base_insurance_object obj
    on obj.id = keys.object_id
left join base_insurance_object_characteristics ch
    on ch.id = keys.characteristics_id
left join condition_summary conditions
    on conditions.characteristics_id = keys.characteristics_id
left join bps_request_ins_task_insurance_object task_link
    on task_link.id = keys.task_object_link_id
left join bps_request_ins_task task
    on task.id = keys.task_id
left join bps_request_ins request
    on request.id = keys.request_id
left join bps_contract contract
    on contract.id = keys.contract_id
left join bps_contract previous_contract
    on previous_contract.id = contract.prevcontract_id
left join bps_contractor policyholder
    on policyholder.id = contract.contractor_id
left join bps_corporate_crm crm
    on crm.id = request.corporate_crm_id
left join base_geo_address address
    on address.id = obj.geo_address_id
left join object_link_profile profile
    on profile.object_id = keys.object_id
),

standardized_result as (
    /* приводим результат к общей структуре двух датасетов */
    select
        case
            when raw.contract_id is null then 'not_linked'
            else 'linked'
        end as row_source,
        case
            when count(raw.contract_id) over (
                partition by raw.object_id
            ) = 0 then 'not_linked'
            when count(raw.contract_id) over (
                partition by raw.object_id
            ) = 1 then 'linked'
            else 'multiple_contracts'
        end as contract_link_status,
        count(raw.contract_id) over (
            partition by raw.object_id
        ) as contract_count,
        (raw.contract_id is not null) as has_contract,
        (
            raw.geo_address_id is not null
            or nullif(btrim(raw.full_address), '') is not null
            or nullif(btrim(raw.original_address), '') is not null
        ) as has_address,
        (raw.insured_sum is not null) as has_target,
        case
            when raw.condition_count is null
              or raw.condition_count = 0
                then 'no_conditions'
            when raw.condition_min_insured_sum is distinct from
                 raw.condition_max_insured_sum
                then 'several_target_values'
            when coalesce(raw.condition_currency_count, 0) > 1
                then 'several_currencies'
            when raw.insured_sum <= 0
                then 'target_is_not_positive'
            when raw.insured_sum is null
                then 'target_is_empty'
            else 'target_is_usable'
        end as target_status,

        raw.contract_id,
        raw.contract_number,
        raw.previous_contract_id,
        raw.root_contract_id,
        raw.request_id,
        raw.task_id,
        raw.task_object_link_id,
        raw.characteristics_id,
        raw.object_id,
        raw.geo_address_id,
        raw.policyholder_id,
        raw.corporate_crm_id,

        raw.as_of_date,
        raw.contract_conclusion_date,
        raw.contract_sign_date,
        raw.contract_start_date,
        raw.contract_end_date,
        raw.contract_status,
        raw.ins_document_type,
        raw.insurance_product,

        case
            when raw.contract_id is not null then
                count(raw.object_id) over (
                    partition by raw.contract_id
                )
        end as real_estate_objects_in_contract,
        raw.object_name,
        raw.object_description,
        raw.object_type,
        raw.elementary_obj_type,
        raw.total_area,
        raw.occupied_area,
        raw.construction_year,
        raw.capital_repair_year,
        raw.floors_count,
        raw.occupied_floor,
        raw.walls_material,
        raw.overlap_material,
        raw.roofing_material,
        raw.ownership_type,
        raw.is_leased,
        raw.insured_components,
        raw.activity_types,
        raw.risk_natures,
        raw.insurance_territory,

        raw.full_address,
        raw.original_address,
        raw.postal_code,
        raw.address_region_id,
        raw.district,
        raw.settlement,
        raw.street,
        raw.house,
        raw.building,
        raw.block,
        raw.flat,
        raw.office,
        raw.fias_code,
        raw.longitude,
        raw.latitude,
        raw.address_dgis_id,

        raw.policyholder_inn,
        raw.policyholder_name,
        raw.policyholder_cdi_id,
        raw.crm_segment,
        raw.crm_macroindustry,
        raw.crm_industry,
        raw.crm_okved,
        raw.business_segment,
        raw.task_industry,
        raw.task_subindustry,

        raw.contract_insured_sum,
        raw.contract_amount_currency,
        raw.task_object_insured_sum,
        raw.task_object_insured_sum_currency,
        raw.condition_min_insured_sum,
        raw.condition_max_insured_sum,
        raw.insured_sum,
        raw.insured_sum_currency,
        raw.condition_currency_count,
        raw.contract_premium,
        raw.insurance_value,
        raw.insurance_value_currency,
        raw.insurance_value_basis,
        raw.is_pledged,
        raw.pledged_value,
        raw.minimum_per_occurrence_limit,
        raw.maximum_per_occurrence_limit,

        raw.previous_contract_number,
        raw.previous_contract_start_date,
        raw.previous_contract_end_date,

        raw.condition_count,
        raw.characteristics_version_number,
        raw.characteristics_version_start_date,
        raw.characteristics_version_end_date,
        raw.characteristics_version_is_active,
        raw.object_characteristics_json,
        raw.task_type,
        raw.task_status,
        raw.ins_refuse,
        raw.source_json__base_insurance_object,
        raw.source_json__base_insurance_object_characteristics,
        raw.source_json__base_insurance_object_conditions,
        raw.source_json__base_geo_address,
        raw.source_json__bps_request_ins_task_insurance_object,
        raw.source_json__bps_request_ins_task,
        raw.source_json__bps_request_ins,
        raw.source_json__bps_contract,
        raw.source_json__bps_contract_previous,
        raw.source_json__bps_contractor,
        raw.source_json__bps_corporate_crm
    from raw_result raw
)

select *
from standardized_result
order by
    has_contract desc,
    as_of_date desc nulls last,
    object_id;

"""


In [ ]:
with engine.connect() as connection:
    expanded_df = pd.read_sql_query(text(expanded_sql), connection)

print('Строк:', len(expanded_df))
print('Колонок:', len(expanded_df.columns))
print('Данные Сферы загружены')

# разворачиваем все поля исходных таблиц в отдельные колонки
def parse_json_container(value):
    if value is None or (not isinstance(value, (dict, list)) and pd.isna(value)):
        return None
    if isinstance(value, (dict, list)):
        return value
    if isinstance(value, str):
        try:
            return json.loads(value)
        except json.JSONDecodeError:
            return value
    return value


def one_value_or_json(values):
    clean_values = []
    seen = set()
    for value in values:
        if value is None or (not isinstance(value, (dict, list)) and pd.isna(value)):
            continue
        marker = json.dumps(value, ensure_ascii=False, sort_keys=True, default=str)
        if marker not in seen:
            seen.add(marker)
            clean_values.append(value)
    if not clean_values:
        return None
    if len(clean_values) == 1:
        value = clean_values[0]
        if isinstance(value, (dict, list)):
            return json.dumps(value, ensure_ascii=False, sort_keys=True, default=str)
        return value
    return json.dumps(clean_values, ensure_ascii=False, sort_keys=True, default=str)


json_columns = [
    column for column in expanded_df.columns
    if column.startswith('source_json__')
]

for json_column in json_columns:
    table_name = json_column.removeprefix('source_json__')
    containers = expanded_df[json_column].map(parse_json_container)

    if table_name == 'base_insurance_object_conditions':
        condition_keys = sorted({
            key
            for rows in containers.dropna()
            if isinstance(rows, list)
            for row in rows
            if isinstance(row, dict)
            for key in row
        })
        normalized = pd.DataFrame(index=expanded_df.index)
        for key in condition_keys:
            normalized[f'sphere__{table_name}__{key}'] = containers.map(
                lambda rows: one_value_or_json([
                    row.get(key)
                    for row in rows
                    if isinstance(row, dict)
                ]) if isinstance(rows, list) else None
            )
    else:
        records = containers.map(
            lambda value: value if isinstance(value, dict) else {}
        ).tolist()
        normalized = pd.json_normalize(records, sep='__')
        normalized.index = expanded_df.index
        normalized.columns = [
            f'sphere__{table_name}__{column}'
            for column in normalized.columns
        ]

    expanded_df = pd.concat(
        [expanded_df.drop(columns=[json_column]), normalized],
        axis=1,
    )

if expanded_df.columns.duplicated().any():
    duplicates = expanded_df.columns[expanded_df.columns.duplicated()].tolist()
    raise ValueError('После раскрытия таблиц появились повторные колонки: ' + ', '.join(duplicates))

print('Колонок после раскрытия всех полей Сферы:', len(expanded_df.columns))


# 4. Проверка заполненности


In [ ]:
required_columns = {
    'row_source', 'object_id', 'characteristics_id', 'contract_id',
    'elementary_obj_type', 'has_contract', 'has_address', 'has_target',
    'target_status'
}
missing_columns = sorted(required_columns - set(expanded_df.columns))
if missing_columns:
    raise ValueError('Не найдены ожидаемые колонки: ' + ', '.join(missing_columns))

profile = pd.DataFrame({
    'Показатель': [
        'Строк',
        'Уникальных объектов',
        'Уникальных характеристик',
        'Уникальных договоров',
        'Строк с договором',
        'Строк с адресом',
        'Строк с target',
        'Строк с пустым типом объекта',
    ],
    'Значение': [
        len(expanded_df),
        expanded_df['object_id'].nunique(dropna=True),
        expanded_df['characteristics_id'].nunique(dropna=True),
        expanded_df['contract_id'].nunique(dropna=True),
        expanded_df['has_contract'].fillna(False).astype(bool).sum(),
        expanded_df['has_address'].fillna(False).astype(bool).sum(),
        expanded_df['has_target'].fillna(False).astype(bool).sum(),
        expanded_df['elementary_obj_type'].fillna('').str.strip().eq('').sum(),
    ],
})

display(profile)

In [ ]:
display(expanded_df['row_source'].fillna('empty').value_counts(dropna=False))
display(expanded_df['elementary_obj_type'].fillna('empty').value_counts(dropna=False))
display(expanded_df['target_status'].fillna('empty').value_counts(dropna=False))

# 5. Получение ФИАС через CDI

Каждый уникальный `full_address` передаётся без изменений в `DM_MOTOR.F_GET_CDI_ADDR_BY_TEXT`.

Адрес разбирает сам CDI. Ноутбук не удаляет квартиру, не меняет регистр и не переставляет части адреса.

Если CDI вернул один `HOUSE_FIAS_ID`, связь считается однозначной. Если разных ФИАС несколько, ни один из них автоматически не выбирается.


In [ ]:
# готовим уникальные full_address для поиска CDI
sphere_with_row_id = expanded_df.copy()
sphere_with_row_id.insert(
    0,
    'sphere_row_id',
    range(1, len(sphere_with_row_id) + 1),
)
sphere_with_row_id['source_address'] = (
    sphere_with_row_id['full_address'].astype('string')
)
empty_address = (
    sphere_with_row_id['source_address'].str.strip().eq('')
)
sphere_with_row_id.loc[empty_address, 'source_address'] = pd.NA

unique_addresses = (
    sphere_with_row_id[['source_address']]
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
)
unique_addresses.insert(
    0,
    'address_lookup_id',
    range(1, len(unique_addresses) + 1),
)

address_records = [
    {
        'address_lookup_id': int(row.address_lookup_id),
        'full_address': str(row.source_address),
    }
    for row in unique_addresses.itertuples(index=False)
]
address_json = json.dumps(address_records, ensure_ascii=False)

print('Объектов:', len(sphere_with_row_id))
print('Объектов с full_address:', sphere_with_row_id['source_address'].notna().sum())
print('Уникальных full_address:', len(unique_addresses))


In [ ]:
# full_address передаётся в функцию CDI без изменений
CDI_SCHEMA = credentials.get('CDI_SCHEMA', 'DM_MOTOR').upper()
CDI_TEXT_FUNCTION = credentials.get(
    'CDI_TEXT_FUNCTION',
    'F_GET_CDI_ADDR_BY_TEXT',
).upper()

for value, label in [
    (CDI_SCHEMA, 'CDI_SCHEMA'),
    (CDI_TEXT_FUNCTION, 'CDI_TEXT_FUNCTION'),
]:
    if not value.replace('_', '').isalnum():
        raise ValueError(f'Некорректное значение {label}')

cdi_function_candidates = [
    f'select d.* from {CDI_SCHEMA}.{CDI_TEXT_FUNCTION}(:address_text) d',
    f'select d.* from table({CDI_SCHEMA}.{CDI_TEXT_FUNCTION}(:address_text)) d',
    f'select d.* from {CDI_TEXT_FUNCTION}(:address_text) d',
    f'select d.* from table({CDI_TEXT_FUNCTION}(:address_text)) d',
]

# в Oracle форма вызова функции может зависеть от версии и прав
cdi_function_sql = cdi_function_candidates[0]
if not unique_addresses.empty:
    test_address = unique_addresses.iloc[0]['source_address']
    call_errors = []
    cdi_function_sql = None
    with khd_connection.cursor() as cursor:
        for candidate_sql in cdi_function_candidates:
            try:
                cursor.execute(candidate_sql, address_text=test_address)
                cursor.fetchmany(1)
                cdi_function_sql = candidate_sql
                break
            except oracledb.Error as error:
                call_errors.append(str(error))

    if cdi_function_sql is None:
        error_details = '\n'.join(
            f'{number}. {message}'
            for number, message in enumerate(call_errors, start=1)
        )
        raise RuntimeError(
            'Не удалось вызвать CDI. '
            f'Подключение: user={KHD_USER}, service={KHD_SERVICE_NAME}.\n'
            f'Ошибки Oracle:\n{error_details}'
        )

print('Вызов CDI:', cdi_function_sql)

cdi_raw_records = []
cdi_error_records = []
cdi_result_columns = None

with khd_connection.cursor() as cursor:
    for row_number, row in enumerate(
        unique_addresses.itertuples(index=False),
        start=1,
    ):
        try:
            cursor.execute(
                cdi_function_sql,
                address_text=row.source_address,
            )
            result_columns = [
                str(column[0]).lower()
                for column in cursor.description
            ]
            if cdi_result_columns is None:
                cdi_result_columns = result_columns

            while True:
                batch = cursor.fetchmany(100)
                if not batch:
                    break
                for values in batch:
                    record = dict(zip(result_columns, values))
                    record['address_lookup_id'] = int(row.address_lookup_id)
                    record['sphere_full_address'] = row.source_address
                    cdi_raw_records.append(record)

        except oracledb.Error as error:
            error_text = str(error)
            cdi_error_records.append({
                'address_lookup_id': int(row.address_lookup_id),
                'error': error_text,
            })

            # такая ошибка означает, что функция недоступна
            if any(code in error_text for code in [
                'ORA-00904', 'ORA-00942', 'ORA-06550', 'ORA-01031'
            ]):
                raise RuntimeError(
                    'Функция CDI недоступна: '
                    f'{CDI_SCHEMA}.{CDI_TEXT_FUNCTION}. '
                    'Ошибка Oracle: ' + error_text
                ) from error

        if row_number % 100 == 0:
            print(
                'Обработано адресов:',
                row_number,
                'из',
                len(unique_addresses),
            )

cdi_raw_df = pd.DataFrame(cdi_raw_records)
cdi_errors_df = pd.DataFrame(cdi_error_records)

# если адресов нет, сохраняем объекты без связи с CDI
if unique_addresses.empty:
    cdi_result_columns = ['house_fias_id']

if cdi_result_columns is None:
    raise RuntimeError('CDI не вернул структуру результата')

# в разных версиях CDI колонка ФИАС дома может называться по-разному
house_fias_column = next(
    (
        column
        for column in [
            'house_fias_id',
            'fias_id_house',
            'fias_house_id',
        ]
        if column in cdi_result_columns
    ),
    None,
)
if house_fias_column is None:
    raise ValueError(
        'CDI не вернул колонку ФИАС дома. '
        'Колонки: ' + ', '.join(cdi_result_columns)
    )

if cdi_raw_df.empty:
    cdi_raw_df = pd.DataFrame(
        columns=[
            'address_lookup_id',
            'sphere_full_address',
            house_fias_column,
        ]
    )

cdi_raw_df['house_fias_for_match'] = (
    cdi_raw_df[house_fias_column]
    .astype('string')
    .str.strip()
    .replace('', pd.NA)
)

cdi_summary = (
    cdi_raw_df.groupby('address_lookup_id', dropna=False)
    .agg(
        cdi_address_candidate_count=('address_lookup_id', 'size'),
        cdi_house_fias_candidate_count=(
            'house_fias_for_match',
            'nunique',
        ),
    )
    .reset_index()
)

unique_fias = (
    cdi_summary['cdi_house_fias_candidate_count'].eq(1)
)
unique_address_ids = set(
    cdi_summary.loc[unique_fias, 'address_lookup_id']
)
cdi_chosen = (
    cdi_raw_df.loc[
        cdi_raw_df['address_lookup_id'].isin(unique_address_ids)
        & cdi_raw_df['house_fias_for_match'].notna()
    ]
    .drop_duplicates(['address_lookup_id', 'house_fias_for_match'])
    .drop_duplicates('address_lookup_id')
    [['address_lookup_id', 'house_fias_for_match']]
    .rename(columns={'house_fias_for_match': 'cdi_fias_id_house'})
)

cdi_lookup_df = (
    unique_addresses
    .rename(columns={'source_address': 'sphere_full_address'})
    .merge(cdi_summary, on='address_lookup_id', how='left')
    .merge(cdi_chosen, on='address_lookup_id', how='left')
)

for column in [
    'cdi_address_candidate_count',
    'cdi_house_fias_candidate_count',
]:
    cdi_lookup_df[column] = (
        cdi_lookup_df[column].fillna(0).astype('int64')
    )

cdi_lookup_df['cdi_is_unique_match'] = (
    cdi_lookup_df['cdi_house_fias_candidate_count'].eq(1).astype('int64')
)
cdi_lookup_df['cdi_match_status'] = 'not_found'
cdi_lookup_df.loc[
    cdi_lookup_df['cdi_house_fias_candidate_count'].eq(1),
    'cdi_match_status',
] = 'unique_house_fias'
cdi_lookup_df.loc[
    cdi_lookup_df['cdi_house_fias_candidate_count'].gt(1),
    'cdi_match_status',
] = 'ambiguous_house_fias'

# ошибку вызова CDI не путаем с отсутствием адреса
if not cdi_errors_df.empty:
    error_address_ids = set(cdi_errors_df['address_lookup_id'])
    error_mask = cdi_lookup_df['address_lookup_id'].isin(
        error_address_ids
    )
    cdi_lookup_df.loc[error_mask, 'cdi_match_status'] = 'lookup_error'
    cdi_lookup_df.loc[error_mask, 'cdi_is_unique_match'] = 0
    cdi_lookup_df.loc[error_mask, 'cdi_fias_id_house'] = pd.NA

cdi_lookup_df['cdi_match_level'] = pd.NA
cdi_lookup_df.loc[
    cdi_lookup_df['cdi_is_unique_match'].eq(1),
    'cdi_match_level',
] = 'full_address_via_cdi'

print('Адресов передано в CDI:', len(unique_addresses))
print('Адресов с одним ФИАС дома:', cdi_lookup_df['cdi_is_unique_match'].sum())
print('Ошибок обработки адреса:', len(cdi_errors_df))
print(cdi_lookup_df['cdi_match_status'].value_counts(dropna=False))


In [ ]:
# возвращаем результат CDI ко всем объектам с таким full_address
address_result = unique_addresses.merge(
    cdi_lookup_df,
    on='address_lookup_id',
    how='left',
    validate='one_to_one',
)

cdi_address_df = sphere_with_row_id.merge(
    address_result.drop(columns=['source_address']),
    left_on='source_address',
    right_on='sphere_full_address',
    how='left',
    validate='many_to_one',
)

for column in [
    'cdi_address_candidate_count',
    'cdi_house_fias_candidate_count',
    'cdi_is_unique_match',
]:
    cdi_address_df[column] = (
        pd.to_numeric(cdi_address_df[column], errors='coerce')
        .fillna(0)
        .astype('int64')
    )

cdi_address_df['cdi_match_status'] = (
    cdi_address_df['cdi_match_status'].fillna('no_source_address')
)
cdi_address_df['cdi_address_match_status'] = cdi_address_df['cdi_match_status']
cdi_address_df['cdi_fias_id_flat'] = pd.NA
cdi_address_df['cdi_flat_is_unique_match'] = 0
cdi_address_df['cdi_flat_match_status'] = 'not_checked'
cdi_address_df['cdi_flat_fias_candidate_count'] = 0
cdi_address_df['sphere_has_premise_in_address'] = (
    cdi_address_df['source_address']
    .astype('string')
    .str.contains(
        r'(?:^|[,;\s])(?:квартира|кв\.?|офис|помещение|пом\.?|комната|комн\.?|апартамент)',
        case=False,
        regex=True,
        na=False,
    )
)

candidate_rows = cdi_raw_df.copy()

print('Строк после CDI:', len(cdi_address_df))
print(cdi_address_df['cdi_match_status'].value_counts(dropna=False))


# 6. Поиск здания в ЕГРН

В ЕГРН передаётся только однозначный ФИАС дома, который CDI получил из `full_address`.

Из ЕГРН выбираются только здания. Квартиры, офисы, помещения, сооружения и строения не участвуют. Площадь не используется.


In [ ]:
# готовим ФИАС дома для поиска ЕГРН
egrn_input = cdi_address_df.loc[
    cdi_address_df['cdi_is_unique_match'].eq(1),
    ['sphere_row_id', 'cdi_fias_id_house'],
].copy()


def json_scalar(value):
    if value is None or pd.isna(value):
        return None
    return str(value)


egrn_records = [
    {
        'sphere_row_id': int(row.sphere_row_id),
        'fias_id_house': json_scalar(row.cdi_fias_id_house),
    }
    for row in egrn_input.itertuples(index=False)
]
egrn_json = json.dumps(egrn_records, ensure_ascii=False)

print('Строк передано в поиск ЕГРН:', len(egrn_records))


In [ ]:
egrn_by_fias_sql = r"""
/*
запрос ищет только здания по ФИАС дома
площадь при выборе здания не используется
*/

with sphere_objects as (
    select /*+ materialize */
        s.sphere_row_id,
        trim(s.fias_id_house) as fias_id_house
    from json_table(
        :egrn_json,
        '$[*]'
        columns (
            sphere_row_id number path '$.sphere_row_id',
            fias_id_house varchar2(500) path '$.fias_id_house'
        )
    ) s
),

egrn_raw as (
    select /*+ no_parallel(e) */
        s.sphere_row_id,
        coalesce(
            nullif(trim(e.cadaster), ''),
            'CAD_IND:' || cast(e.cad_ind as varchar2(200))
        ) as egrn_key,
        e.cad_ind,
        e.cadaster,
        e.egrn_address,
        e.square,
        e.measure,
        e.building_type,
        e.oks_type,
        e.oks_purpose,
        e.object_status,
        e.fias_level,
        e.fias_id_house,
        e.row_update_date,
        e.ias_update_date
    from sphere_objects s
    join DM_RISK_AVATAR.EGRN_DATA e
        on e.fias_id_house = s.fias_id_house
    where upper(trim(e.fias_level)) = 'FIAS_HOUSE'
      and lower(trim(e.oks_type)) = 'здание'
      and e.flat is null
      and e.flat2 is null
      and e.office is null
      and e.office2 is null
      and e.room is null
      and e.room2 is null
      and e.compartment1 is null
      and e.compartment2 is null
      and (e.cadaster is not null or e.cad_ind is not null)
),

ranked_versions as (
    select
        e.*,
        row_number() over (
            partition by e.sphere_row_id, e.egrn_key
            order by
                e.row_update_date desc nulls last,
                e.ias_update_date desc nulls last,
                e.cad_ind desc nulls last
        ) as version_number
    from egrn_raw e
),

current_buildings as (
    select e.*
    from ranked_versions e
    where e.version_number = 1
),

buildings_with_count as (
    select
        e.*,
        count(*) over (
            partition by e.sphere_row_id
        ) as candidate_count
    from current_buildings e
),

candidate_summary as (
    select
        e.sphere_row_id,
        max(e.candidate_count) as candidate_count
    from buildings_with_count e
    group by e.sphere_row_id
),

chosen_building as (
    select e.*
    from buildings_with_count e
    where e.candidate_count = 1
)

select /*+ no_parallel */
    s.sphere_row_id as "sphere_row_id",
    s.fias_id_house as "cdi_fias_id_house",
    nvl(summary.candidate_count, 0) as "egrn_address_candidate_count",
    nvl(summary.candidate_count, 0) as "egrn_candidate_count",
    case
        when summary.candidate_count = 1 then 1
        else 0
    end as "egrn_is_unique_match",
    case
        when nvl(summary.candidate_count, 0) = 0 then 'not_found'
        when summary.candidate_count = 1 then 'fias_house_only'
        else 'ambiguous'
    end as "egrn_match_method",
    building.cad_ind as "egrn_cad_ind",
    building.cadaster as "egrn_cadaster",
    building.egrn_address as "egrn_address",
    building.square as "egrn_square",
    building.measure as "egrn_measure",
    building.building_type as "egrn_building_type",
    building.oks_type as "egrn_oks_type",
    building.oks_purpose as "egrn_oks_purpose",
    building.object_status as "egrn_object_status",
    building.fias_level as "egrn_fias_level",
    building.fias_id_house as "egrn_fias_id_house"
from sphere_objects s
left join candidate_summary summary
    on summary.sphere_row_id = s.sphere_row_id
left join chosen_building building
    on building.sphere_row_id = s.sphere_row_id
order by s.sphere_row_id
"""


In [ ]:
# выполняем один запрос к ЕГРН и получаем одну строку на объект Сферы
egrn_base_columns = [
    'sphere_row_id',
    'cdi_fias_id_house',
    'egrn_address_candidate_count',
    'egrn_candidate_count',
    'egrn_is_unique_match',
    'egrn_match_method',
    'egrn_cad_ind',
    'egrn_cadaster',
    'egrn_address',
    'egrn_square',
    'egrn_measure',
    'egrn_building_type',
    'egrn_oks_type',
    'egrn_oks_purpose',
    'egrn_object_status',
    'egrn_fias_level',
    'egrn_fias_id_house',
]

khd_schema = KHD_DATA_SCHEMA.upper()
if not khd_schema.replace('_', '').isalnum():
    raise ValueError('Некорректное имя схемы КХД')

invalid_egrn_columns = [
    column for column in egrn_source_columns
    if not column.replace('_', '').isalnum()
]
if invalid_egrn_columns:
    raise ValueError(
        'В EGRN_DATA найдены некорректные имена колонок: '
        + ', '.join(invalid_egrn_columns)
    )

# короткие псевдонимы нужны Oracle, чтобы вернуть все поля EGRN_DATA
egrn_oracle_aliases = {
    column: f'ER{index:03d}'
    for index, column in enumerate(egrn_source_columns, start=1)
}
egrn_raw_columns = [
    f'egrn_raw__{column.lower()}'
    for column in egrn_source_columns
]

if egrn_records:
    egrn_query = egrn_by_fias_sql.replace(
        'DM_RISK_AVATAR.',
        f'{khd_schema}.',
    )

    raw_fields_sql = ',\n        '.join(
        f'e."{column}" as "{egrn_oracle_aliases[column]}"'
        for column in egrn_source_columns
    )
    egrn_query = egrn_query.replace(
        '        s.sphere_row_id,\n        coalesce(',
        '        s.sphere_row_id,\n        '
        + raw_fields_sql
        + ',\n        coalesce(',
        1,
    )

    raw_result_sql = ',\n    '.join(
        f'building."{egrn_oracle_aliases[column]}" '
        f'as "{egrn_oracle_aliases[column]}"'
        for column in egrn_source_columns
    )
    egrn_query = egrn_query.replace(
        '    building.cad_ind as "egrn_cad_ind",',
        '    ' + raw_result_sql
        + ',\n    building.cad_ind as "egrn_cad_ind",',
        1,
    )

    with khd_connection.cursor() as cursor:
        cursor.execute('alter session disable parallel query')
        egrn_json_bind = cursor.var(oracledb.DB_TYPE_CLOB)
        egrn_json_bind.setvalue(0, egrn_json)
        cursor.execute(egrn_query, egrn_json=egrn_json_bind)
        result_columns = [
            str(column[0]).lower()
            for column in cursor.description
        ]
        result_rows = cursor.fetchall()

    egrn_lookup_df = pd.DataFrame(
        result_rows,
        columns=result_columns,
    )
    egrn_lookup_df = egrn_lookup_df.rename(columns={
        alias.lower(): f'egrn_raw__{column.lower()}'
        for column, alias in egrn_oracle_aliases.items()
    })
else:
    egrn_lookup_df = pd.DataFrame(
        columns=egrn_base_columns + egrn_raw_columns
    )

egrn_expected_columns = egrn_base_columns + egrn_raw_columns
missing_egrn_columns = sorted(
    set(egrn_expected_columns) - set(egrn_lookup_df.columns)
)
if missing_egrn_columns:
    raise ValueError(
        'ЕГРН не вернул ожидаемые колонки: '
        + ', '.join(missing_egrn_columns)
    )
if egrn_lookup_df['sphere_row_id'].duplicated().any():
    raise ValueError('ЕГРН вернул несколько итоговых строк для объекта Сферы')

expanded_egrn_df = cdi_address_df.merge(
    egrn_lookup_df[egrn_expected_columns],
    on='sphere_row_id',
    how='left',
    validate='one_to_one',
    suffixes=('', '_egrn_result'),
)
expanded_egrn_df['egrn_is_unique_match'] = (
    pd.to_numeric(
        expanded_egrn_df['egrn_is_unique_match'],
        errors='coerce',
    )
    .fillna(0)
    .astype('int64')
)

expanded_egrn_df['pipeline_match_status'] = 'not_linked'
expanded_egrn_df.loc[
    expanded_egrn_df['cdi_is_unique_match'].eq(1),
    'pipeline_match_status',
] = 'cdi_address_linked'
expanded_egrn_df.loc[
    expanded_egrn_df['egrn_is_unique_match'].eq(1),
    'pipeline_match_status',
] = 'egrn_linked'

expanded_egrn_df['connection_method'] = 'not_found'
expanded_egrn_df.loc[
    expanded_egrn_df['cdi_match_status'].eq('no_source_address'),
    'connection_method',
] = 'no_source_address'
expanded_egrn_df.loc[
    expanded_egrn_df['cdi_match_status'].eq('not_found'),
    'connection_method',
] = 'cdi_address_not_found'
expanded_egrn_df.loc[
    expanded_egrn_df['cdi_match_status'].eq('lookup_error'),
    'connection_method',
] = 'cdi_lookup_error'
expanded_egrn_df.loc[
    expanded_egrn_df['cdi_match_status'].eq('ambiguous_house_fias'),
    'connection_method',
] = 'cdi_house_fias_ambiguous'
expanded_egrn_df.loc[
    expanded_egrn_df['cdi_is_unique_match'].eq(1),
    'connection_method',
] = 'cdi_house_fias_found_egrn_not_found'
expanded_egrn_df.loc[
    expanded_egrn_df['egrn_match_method'].eq('ambiguous'),
    'connection_method',
] = 'egrn_house_ambiguous'
expanded_egrn_df.loc[
    expanded_egrn_df['egrn_match_method'].eq('fias_house_only'),
    'connection_method',
] = 'egrn_house_by_fias'
expanded_egrn_df['connection_is_unique'] = (
    expanded_egrn_df['egrn_is_unique_match'].eq(1).astype('int64')
)

if len(expanded_egrn_df) != len(expanded_df):
    raise ValueError('После CDI и ЕГРН изменилось количество строк датасета')

print('Строк в итоговом датасете:', len(expanded_egrn_df))
print('Однозначно присоединено зданий:', int(expanded_egrn_df['connection_is_unique'].sum()))


# 7. Проверка результата

- `connection_is_unique = 1` — найдено ровно одно здание ЕГРН;
- `connection_method = egrn_house_by_fias` — здание присоединено;
- `connection_method = egrn_house_ambiguous` — по одному ФИАС найдено несколько зданий, поэтому данные не присоединены;
- `connection_method = cdi_house_fias_found_egrn_not_found` — CDI вернул ФИАС дома, но здание ЕГРН не найдено.


In [ ]:
quality_profile = pd.DataFrame({
    'Показатель': [
        'Строк исходного датасета',
        'Строк итогового датасета',
        'Строк без full_address',
        'CDI вернул один ФИАС дома',
        'ЕГРН присоединён однозначно',
        'В ЕГРН найдено несколько зданий',
        'Здание ЕГРН не найдено',
    ],
    'Значение': [
        len(expanded_df),
        len(expanded_egrn_df),
        expanded_egrn_df['cdi_match_status'].eq('no_source_address').sum(),
        expanded_egrn_df['cdi_is_unique_match'].sum(),
        expanded_egrn_df['egrn_is_unique_match'].sum(),
        expanded_egrn_df['egrn_match_method'].eq('ambiguous').sum(),
        expanded_egrn_df['egrn_match_method'].eq('not_found').sum(),
    ],
})

display(quality_profile)
display(expanded_egrn_df['connection_method'].value_counts(dropna=False))


# 8. Статистический паспорт

Для анализа используется результат после соединения Сферы с ЕГРН. Каждая колонка результата попадёт в паспорт отдельной строкой.


In [ ]:
df = expanded_egrn_df.copy()
df.columns = [str(column).strip() for column in df.columns]

results_dir = OUTPUT_DIR
output_file = results_dir / 'паспорт_всех_признаков_расширенный.csv'

# словарь может лежать рядом с ноутбуком или в папке материалов проекта
dictionary_candidates = [
    NOTEBOOK_DIR / 'колонки_таблиц.xlsx',
    PROJECT_ROOT / 'колонки_таблиц.xlsx',
    PROJECT_ROOT / 'МАТЕРИАЛЫ_ПРОЕКТА' / '00_входные_материалы' / 'колонки_таблиц.xlsx',
]
dictionary_file = next(
    (path for path in dictionary_candidates if path.exists()),
    None,
)

target_column = 'insured_sum'
csv_separator = ';'
csv_encoding = 'utf-8-sig'
top_values_limit = 10

if target_column not in df.columns:
    raise ValueError(f'В датасете нет целевой колонки {target_column}')

print('Строк для анализа:', len(df))
print('Колонок для анализа:', len(df.columns))
print('Словарь колонок:', dictionary_file or 'не найден')
print('Итоговый паспорт:', output_file)


In [ ]:
# словарь нужен для русских названий таблиц и колонок
dictionary_rows = pd.DataFrame()

if dictionary_file is not None:
    dictionary_parts = []
    excel_file = pd.ExcelFile(dictionary_file)
    for sheet_name in ['Сфера', 'КХД 1.0']:
        if sheet_name not in excel_file.sheet_names:
            continue
        part = pd.read_excel(dictionary_file, sheet_name=sheet_name)
        part['SOURCE_SYSTEM'] = sheet_name
        dictionary_parts.append(part)

    if dictionary_parts:
        dictionary_rows = pd.concat(dictionary_parts, ignore_index=True)
        dictionary_rows.columns = [
            str(column).strip().upper()
            for column in dictionary_rows.columns
        ]
        required_dictionary_columns = {'TABLE_NAME', 'COLUMN_NAME'}
        missing_dictionary_columns = sorted(
            required_dictionary_columns - set(dictionary_rows.columns)
        )
        if missing_dictionary_columns:
            raise ValueError(
                'В словаре нет колонок: '
                + ', '.join(missing_dictionary_columns)
            )
        dictionary_rows['TABLE_NAME_KEY'] = (
            dictionary_rows['TABLE_NAME'].astype('string').str.upper()
        )
        dictionary_rows['COLUMN_NAME_KEY'] = (
            dictionary_rows['COLUMN_NAME'].astype('string').str.lower()
        )

print('Строк в словаре:', len(dictionary_rows))


In [ ]:
technical_columns = {
    'sphere_row_id',
    'row_source',
    'contract_link_status',
    'contract_count',
    'has_contract',
    'has_address',
    'has_target',
    'target_status',
    'source_address',
    'input_address',
    'cdi_house_fias_candidate_count',
    'cdi_match_status',
    'cdi_is_unique_match',
    'egrn_address_candidate_count',
    'egrn_candidate_count',
    'egrn_is_unique_match',
    'egrn_match_method',
    'pipeline_match_status',
    'external_snapshot_at',
}

identifier_columns = {
    'contract_id',
    'contract_number',
    'previous_contract_id',
    'root_contract_id',
    'request_id',
    'task_id',
    'task_object_link_id',
    'characteristics_id',
    'object_id',
    'geo_address_id',
    'policyholder_id',
    'corporate_crm_id',
    'policyholder_inn',
    'policyholder_cdi_id',
    'policyholder_ogrn',
    'policyholder_kpp',
    'address_dgis_id',
    'cdi_fias_id_house',
    'egrn_cad_ind',
    'egrn_cadaster',
    'egrn_fias_id_house',
}

target_derived_columns = {
    'has_target',
    'target_status',
    'condition_min_insured_sum',
    'condition_max_insured_sum',
    'task_object_insured_sum',
    'contract_insured_sum',
}

sensitive_pattern = re.compile(
    r'(address|адрес|inn|ogrn|kpp|cadaster|contract_number|'
    r'policyholder_name|company_name|description|comment|json|'
    r'phone|email|(^|_)id($|_))',
    flags=re.IGNORECASE,
)

date_pattern = re.compile(
    r'(date|time|timestamp|d_create|d_change|d_delete|_start|_end|as_of)',
    flags=re.IGNORECASE,
)


def infer_source(column):
    if column in technical_columns:
        return 'Технический'
    if column.startswith('egrn_'):
        return 'ЕГРН'
    if column.startswith('cdi_'):
        return 'CDI'
    return 'Сфера'


def infer_role(column):
    if column == target_column:
        return 'target'
    if column == 'as_of_date':
        return 'split_key'
    if column in technical_columns:
        return 'service'
    if column in identifier_columns or column.endswith('_id'):
        return 'identifier'
    return 'feature'


In [ ]:
# основные поля датасета имеют короткие имена, поэтому для них
# отдельно указываем исходную таблицу и исходную колонку
exact_source_map = {
    'contract_id': ('BPS_CONTRACT', 'ID'),
    'contract_number': ('BPS_CONTRACT', 'N_CONTRACT'),
    'previous_contract_id': ('BPS_CONTRACT', 'PREVIOUS_CONTRACT_ID'),
    'request_id': ('BPS_REQUEST_INS', 'ID'),
    'task_id': ('BPS_REQUEST_INS_TASK', 'ID'),
    'task_object_link_id': ('BPS_REQUEST_INS_TASK_INSURANCE_OBJECT', 'ID'),
    'characteristics_id': ('BASE_INSURANCE_OBJECT_CHARACTERISTICS', 'ID'),
    'object_id': ('BASE_INSURANCE_OBJECT', 'ID'),
    'geo_address_id': ('BASE_GEO_ADDRESS', 'ID'),
    'policyholder_id': ('BPS_CONTRACTOR', 'ID'),
    'corporate_crm_id': ('BPS_CORPORATE_CRM', 'ID'),
    'insured_sum': ('BASE_INSURANCE_OBJECT_CONDITIONS', 'INSURED_SUM'),
    'insured_sum_currency': ('BASE_INSURANCE_OBJECT_CONDITIONS', 'INSURED_SUM_CURRENCY'),
    'condition_min_insured_sum': ('BASE_INSURANCE_OBJECT_CONDITIONS', 'INSURED_SUM'),
    'condition_max_insured_sum': ('BASE_INSURANCE_OBJECT_CONDITIONS', 'INSURED_SUM'),
    'full_address': ('BASE_GEO_ADDRESS', 'FULL_ADDRESS'),
    'original_address': ('BASE_INSURANCE_OBJECT', 'ORIGINAL_ADDRESS'),
    'object_name': ('BASE_INSURANCE_OBJECT', 'OBJ_NAME'),
    'object_description': ('BASE_INSURANCE_OBJECT', 'DESCRIPTION'),
    'object_type': ('BASE_INSURANCE_OBJECT', 'OBJ_TYPE'),
    'elementary_obj_type': ('BASE_INSURANCE_OBJECT', 'ELEMENTARY_OBJ_TYPE'),
    'total_area': ('BASE_INSURANCE_OBJECT_CHARACTERISTICS', 'CHARACTERISTICS'),
    'policyholder_inn': ('BPS_CONTRACTOR', 'INN'),
    'policyholder_name': ('BPS_CONTRACTOR', 'COMPANY_NAME_SHORT'),
    'crm_segment': ('BPS_CORPORATE_CRM', 'SEGMENT'),
    'crm_macroindustry': ('BPS_CORPORATE_CRM', 'MACROINDUSTRY'),
    'crm_industry': ('BPS_CORPORATE_CRM', 'INDUSTRY'),
    'crm_okved': ('BPS_CORPORATE_CRM', 'OKVED'),
    'task_type': ('BPS_REQUEST_INS_TASK', 'TASK_TYPE'),
    'task_status': ('BPS_REQUEST_INS_TASK', 'STATUS'),
    'ins_document_type': ('BPS_REQUEST_INS_TASK', 'INS_DOCUMENT_TYPE'),
    'ins_refuse': ('BPS_REQUEST_INS_TASK', 'INS_REFUSE'),
    'contract_insured_sum': ('BPS_REQUEST_INS_TASK', 'TOTAL_INS_CONTRACT_AMOUNT'),
    'contract_premium': ('BPS_REQUEST_INS_TASK', 'TOTAL_INS_CONTRACT_PREMIUM'),
    'task_object_insured_sum': ('BPS_REQUEST_INS_TASK_INSURANCE_OBJECT', 'INSURED_SUM'),
    'insurance_value': ('BASE_INSURANCE_OBJECT_CHARACTERISTICS', 'INSURANCE_VALUE'),
    'pledged_value': ('BASE_INSURANCE_OBJECT_CHARACTERISTICS', 'PLEDGED_VALUE'),
}
exact_source_map.update({'contract_id': ('BPS_CONTRACT', 'contract_id'), 'contract_number': ('BPS_CONTRACT', 'contract_number'), 'previous_contract_id': ('BPS_CONTRACT', 'previous_contract_id'), 'root_contract_id': ('BPS_CONTRACT', 'root_contract_id'), 'contract_conclusion_date': ('BPS_CONTRACT', 'contract_conclusion_date'), 'contract_sign_date': ('BPS_CONTRACT', 'contract_sign_date'), 'contract_start_date': ('BPS_CONTRACT', 'contract_start_date'), 'contract_end_date': ('BPS_CONTRACT', 'contract_end_date'), 'contract_status': ('BPS_CONTRACT', 'contract_status'), 'previous_contract_number': ('BPS_CONTRACT', 'previous_contract_number'), 'previous_contract_start_date': ('BPS_CONTRACT', 'previous_contract_start_date'), 'previous_contract_end_date': ('BPS_CONTRACT', 'previous_contract_end_date'), 'request_id': ('BPS_REQUEST_INS', 'request_id'), 'corporate_crm_id': ('BPS_REQUEST_INS', 'corporate_crm_id'), 'insurance_product': ('BPS_REQUEST_INS', 'insurance_product'), 'business_segment': ('BPS_REQUEST_INS', 'business_segment'), 'task_id': ('BPS_REQUEST_INS_TASK', 'task_id'), 'ins_document_type': ('BPS_REQUEST_INS_TASK', 'ins_document_type'), 'task_industry': ('BPS_REQUEST_INS_TASK', 'task_industry'), 'task_subindustry': ('BPS_REQUEST_INS_TASK', 'task_subindustry'), 'contract_insured_sum': ('BPS_REQUEST_INS_TASK', 'contract_insured_sum'), 'contract_amount_currency': ('BPS_REQUEST_INS_TASK', 'contract_amount_currency'), 'contract_premium': ('BPS_REQUEST_INS_TASK', 'contract_premium'), 'task_type': ('BPS_REQUEST_INS_TASK', 'task_type'), 'task_status': ('BPS_REQUEST_INS_TASK', 'task_status'), 'ins_refuse': ('BPS_REQUEST_INS_TASK', 'ins_refuse'), 'task_object_link_id': ('BPS_REQUEST_INS_TASK_INSURANCE_OBJECT', 'task_object_link_id'), 'task_object_insured_sum': ('BPS_REQUEST_INS_TASK_INSURANCE_OBJECT', 'task_object_insured_sum'), 'task_object_insured_sum_currency': ('BPS_REQUEST_INS_TASK_INSURANCE_OBJECT', 'task_object_insured_sum_currency'), 'object_id': ('BASE_INSURANCE_OBJECT', 'object_id'), 'geo_address_id': ('BASE_INSURANCE_OBJECT', 'geo_address_id'), 'object_name': ('BASE_INSURANCE_OBJECT', 'object_name'), 'object_description': ('BASE_INSURANCE_OBJECT', 'object_description'), 'object_type': ('BASE_INSURANCE_OBJECT', 'object_type'), 'elementary_obj_type': ('BASE_INSURANCE_OBJECT', 'elementary_obj_type'), 'original_address': ('BASE_INSURANCE_OBJECT', 'original_address'), 'characteristics_id': ('BASE_INSURANCE_OBJECT_CHARACTERISTICS', 'characteristics_id'), 'total_area': ('BASE_INSURANCE_OBJECT_CHARACTERISTICS', 'total_area'), 'occupied_area': ('BASE_INSURANCE_OBJECT_CHARACTERISTICS', 'occupied_area'), 'construction_year': ('BASE_INSURANCE_OBJECT_CHARACTERISTICS', 'construction_year'), 'capital_repair_year': ('BASE_INSURANCE_OBJECT_CHARACTERISTICS', 'capital_repair_year'), 'floors_count': ('BASE_INSURANCE_OBJECT_CHARACTERISTICS', 'floors_count'), 'occupied_floor': ('BASE_INSURANCE_OBJECT_CHARACTERISTICS', 'occupied_floor'), 'walls_material': ('BASE_INSURANCE_OBJECT_CHARACTERISTICS', 'walls_material'), 'overlap_material': ('BASE_INSURANCE_OBJECT_CHARACTERISTICS', 'overlap_material'), 'roofing_material': ('BASE_INSURANCE_OBJECT_CHARACTERISTICS', 'roofing_material'), 'ownership_type': ('BASE_INSURANCE_OBJECT_CHARACTERISTICS', 'ownership_type'), 'is_leased': ('BASE_INSURANCE_OBJECT_CHARACTERISTICS', 'is_leased'), 'insured_components': ('BASE_INSURANCE_OBJECT_CHARACTERISTICS', 'insured_components'), 'activity_types': ('BASE_INSURANCE_OBJECT_CHARACTERISTICS', 'activity_types'), 'risk_natures': ('BASE_INSURANCE_OBJECT_CHARACTERISTICS', 'risk_natures'), 'insurance_territory': ('BASE_INSURANCE_OBJECT_CHARACTERISTICS', 'insurance_territory'), 'insurance_value': ('BASE_INSURANCE_OBJECT_CHARACTERISTICS', 'insurance_value'), 'insurance_value_currency': ('BASE_INSURANCE_OBJECT_CHARACTERISTICS', 'insurance_value_currency'), 'insurance_value_basis': ('BASE_INSURANCE_OBJECT_CHARACTERISTICS', 'insurance_value_basis'), 'is_pledged': ('BASE_INSURANCE_OBJECT_CHARACTERISTICS', 'is_pledged'), 'pledged_value': ('BASE_INSURANCE_OBJECT_CHARACTERISTICS', 'pledged_value'), 'characteristics_version_number': ('BASE_INSURANCE_OBJECT_CHARACTERISTICS', 'characteristics_version_number'), 'characteristics_version_start_date': ('BASE_INSURANCE_OBJECT_CHARACTERISTICS', 'characteristics_version_start_date'), 'characteristics_version_end_date': ('BASE_INSURANCE_OBJECT_CHARACTERISTICS', 'characteristics_version_end_date'), 'characteristics_version_is_active': ('BASE_INSURANCE_OBJECT_CHARACTERISTICS', 'characteristics_version_is_active'), 'object_characteristics_json': ('BASE_INSURANCE_OBJECT_CHARACTERISTICS', 'object_characteristics_json'), 'condition_count': ('BASE_INSURANCE_OBJECT_CONDITIONS', 'condition_count'), 'condition_currency_count': ('BASE_INSURANCE_OBJECT_CONDITIONS', 'condition_currency_count'), 'condition_min_insured_sum': ('BASE_INSURANCE_OBJECT_CONDITIONS', 'condition_min_insured_sum'), 'condition_max_insured_sum': ('BASE_INSURANCE_OBJECT_CONDITIONS', 'condition_max_insured_sum'), 'insured_sum': ('BASE_INSURANCE_OBJECT_CONDITIONS', 'insured_sum'), 'insured_sum_currency': ('BASE_INSURANCE_OBJECT_CONDITIONS', 'insured_sum_currency'), 'minimum_per_occurrence_limit': ('BASE_INSURANCE_OBJECT_CONDITIONS', 'minimum_per_occurrence_limit'), 'maximum_per_occurrence_limit': ('BASE_INSURANCE_OBJECT_CONDITIONS', 'maximum_per_occurrence_limit'), 'full_address': ('BASE_GEO_ADDRESS', 'full_address'), 'postal_code': ('BASE_GEO_ADDRESS', 'postal_code'), 'address_region_id': ('BASE_GEO_ADDRESS', 'address_region_id'), 'district': ('BASE_GEO_ADDRESS', 'district'), 'settlement': ('BASE_GEO_ADDRESS', 'settlement'), 'street': ('BASE_GEO_ADDRESS', 'street'), 'house': ('BASE_GEO_ADDRESS', 'house'), 'building': ('BASE_GEO_ADDRESS', 'building'), 'block': ('BASE_GEO_ADDRESS', 'block'), 'flat': ('BASE_GEO_ADDRESS', 'flat'), 'office': ('BASE_GEO_ADDRESS', 'office'), 'fias_code': ('BASE_GEO_ADDRESS', 'fias_code'), 'longitude': ('BASE_GEO_ADDRESS', 'longitude'), 'latitude': ('BASE_GEO_ADDRESS', 'latitude'), 'address_dgis_id': ('BASE_GEO_ADDRESS', 'address_dgis_id'), 'policyholder_id': ('BPS_CONTRACTOR', 'policyholder_id'), 'policyholder_inn': ('BPS_CONTRACTOR', 'policyholder_inn'), 'policyholder_name': ('BPS_CONTRACTOR', 'policyholder_name'), 'policyholder_cdi_id': ('BPS_CONTRACTOR', 'policyholder_cdi_id'), 'crm_segment': ('BPS_CORPORATE_CRM', 'crm_segment'), 'crm_macroindustry': ('BPS_CORPORATE_CRM', 'crm_macroindustry'), 'crm_industry': ('BPS_CORPORATE_CRM', 'crm_industry'), 'crm_okved': ('BPS_CORPORATE_CRM', 'crm_okved'), 'as_of_date': ('расчёт в SQL Сферы', 'as_of_date'), 'real_estate_objects_in_contract': ('расчёт в SQL Сферы', 'real_estate_objects_in_contract'), 'sphere_address_for_egrn': ('расчёт соединения с ЕГРН', 'sphere_address_for_egrn'), 'parsed_postal_code': ('расчёт соединения с ЕГРН', 'parsed_postal_code'), 'parsed_region': ('расчёт соединения с ЕГРН', 'parsed_region'), 'parsed_locality': ('расчёт соединения с ЕГРН', 'parsed_locality'), 'parsed_street': ('расчёт соединения с ЕГРН', 'parsed_street'), 'parsed_house': ('расчёт соединения с ЕГРН', 'parsed_house'), 'parsed_korpus': ('расчёт соединения с ЕГРН', 'parsed_korpus'), 'parsed_stroenie': ('расчёт соединения с ЕГРН', 'parsed_stroenie'), 'parsed_landmark': ('расчёт соединения с ЕГРН', 'parsed_landmark'), 'compared_sphere_postal_code': ('расчёт соединения с ЕГРН', 'compared_sphere_postal_code'), 'compared_sphere_region': ('расчёт соединения с ЕГРН', 'compared_sphere_region'), 'compared_sphere_locality': ('расчёт соединения с ЕГРН', 'compared_sphere_locality'), 'compared_sphere_street': ('расчёт соединения с ЕГРН', 'compared_sphere_street'), 'compared_sphere_house': ('расчёт соединения с ЕГРН', 'compared_sphere_house'), 'compared_sphere_korpus': ('расчёт соединения с ЕГРН', 'compared_sphere_korpus'), 'compared_sphere_stroenie': ('расчёт соединения с ЕГРН', 'compared_sphere_stroenie'), 'matched_egrn_postal_code': ('расчёт соединения с ЕГРН', 'matched_egrn_postal_code'), 'matched_egrn_region': ('расчёт соединения с ЕГРН', 'matched_egrn_region'), 'matched_egrn_locality': ('расчёт соединения с ЕГРН', 'matched_egrn_locality'), 'matched_egrn_street': ('расчёт соединения с ЕГРН', 'matched_egrn_street'), 'matched_egrn_house': ('расчёт соединения с ЕГРН', 'matched_egrn_house'), 'matched_egrn_korpus': ('расчёт соединения с ЕГРН', 'matched_egrn_korpus'), 'matched_egrn_stroenie': ('расчёт соединения с ЕГРН', 'matched_egrn_stroenie')})


explicit_russian_names = {
    'sphere_row_id': 'номер строки датасета',
    'row_source': 'источник строки: связанный или несвязанный объект',
    'contract_link_status': 'результат связи объекта с договором',
    'contract_count': 'количество договоров, связанных с объектом',
    'has_contract': 'есть связь с договором',
    'has_address': 'есть адрес объекта',
    'has_target': 'заполнена целевая страховая сумма',
    'target_status': 'результат проверки целевой переменной',
    'contract_id': 'идентификатор договора',
    'contract_number': 'номер договора',
    'request_id': 'идентификатор заявки',
    'task_id': 'идентификатор задачи оформления',
    'task_object_link_id': 'идентификатор связи задачи и объекта',
    'characteristics_id': 'идентификатор версии характеристик объекта',
    'object_id': 'идентификатор объекта страхования',
    'geo_address_id': 'идентификатор адреса объекта',
    'policyholder_id': 'идентификатор страхователя',
    'corporate_crm_id': 'идентификатор карточки клиента в crm',
    'as_of_date': 'дата, на которую формируются признаки',
    'contract_start_date': 'дата начала договора',
    'contract_end_date': 'дата окончания договора',
    'contract_status': 'статус договора',
    'ins_document_type': 'тип страхового документа',
    'insurance_product': 'страховой продукт',
    'object_name': 'название объекта',
    'object_description': 'описание объекта',
    'object_type': 'тип объекта',
    'elementary_obj_type': 'системный подтип объекта',
    'total_area': 'общая площадь объекта',
    'full_address': 'нормализованный полный адрес',
    'original_address': 'адрес, введённый вручную',
    'policyholder_inn': 'инн страхователя',
    'policyholder_name': 'наименование страхователя',
    'crm_segment': 'сегмент клиента',
    'crm_macroindustry': 'макроотрасль клиента',
    'crm_industry': 'отрасль клиента',
    'crm_okved': 'оквэд клиента',
    'insured_sum': 'страховая сумма объекта — целевая переменная',
    'insured_sum_currency': 'валюта страховой суммы объекта',
    'condition_min_insured_sum': 'минимальная страховая сумма в условиях объекта',
    'condition_max_insured_sum': 'максимальная страховая сумма в условиях объекта',
    'task_object_insured_sum': 'страховая сумма объекта из связи с задачей',
    'contract_insured_sum': 'общая страховая сумма договора',
    'contract_premium': 'общая страховая премия договора',
    'insurance_value': 'страховая стоимость объекта',
    'pledged_value': 'залоговая стоимость объекта',
    'source_address': 'адрес Сферы, переданный в CDI',
    'sphere_full_address': 'полный адрес Сферы, переданный в CDI',
    'cdi_fias_id_house': 'ФИАС дома, полученный из CDI',
    'cdi_address_candidate_count': 'количество строк, полученных из CDI',
    'cdi_house_fias_candidate_count': 'количество разных ФИАС дома из CDI',
    'cdi_is_unique_match': 'CDI вернул один ФИАС дома',
    'cdi_match_status': 'результат поиска адреса через CDI',
    'egrn_address_candidate_count': 'количество зданий ЕГРН по ФИАС дома',
    'egrn_candidate_count': 'количество подходящих зданий ЕГРН',
    'egrn_is_unique_match': 'найдено ровно одно здание ЕГРН',
    'egrn_match_method': 'результат поиска здания ЕГРН',
    'egrn_cad_ind': 'внутренний идентификатор записи ЕГРН',
    'egrn_cadaster': 'кадастровый номер здания',
    'egrn_address': 'адрес здания из ЕГРН',
    'egrn_square': 'площадь здания из ЕГРН',
    'egrn_building_type': 'тип строения из ЕГРН',
    'egrn_oks_type': 'тип объекта капитального строительства',
    'egrn_oks_purpose': 'назначение здания из ЕГРН',
    'egrn_object_status': 'статус объекта ЕГРН',
    'egrn_fias_level': 'уровень адреса ЕГРН',
    'egrn_fias_id_house': 'ФИАС дома из ЕГРН',
    'pipeline_match_status': 'результат прохождения цепочки CDI и ЕГРН',
    'connection_method': 'способ или причина соединения с ЕГРН',
    'connection_is_unique': 'признак однозначного соединения с ЕГРН',
}

column_name_translations = {'point__coordinates': 'координаты адреса', 'point__crs__properties__name': 'система координат', 'point__crs__type': 'тип системы координат', 'point__type': 'тип географической точки', 'air_conditioning_ventilation_system': 'система кондиционирования и вентиляции', 'building_housekeeping_compliance': 'соблюдение требований к содержанию здания', 'charging_equipment_violation': 'нарушения при эксплуатации зарядного оборудования', 'columns_material': 'материал колонн', 'construction_year': 'год постройки', 'emergency_diesel_generator_availability': 'наличие аварийного дизель-генератора', 'external_fire_hydrants_count': 'количество наружных пожарных гидрантов', 'facade_insulation_material': 'материал утепления фасада', 'fire_alarm_coverage_percentage': 'доля здания, покрытая пожарной сигнализацией', 'fire_alarm_detectors_type': 'тип датчиков пожарной сигнализации', 'fire_alarm_monitoring_destination': 'куда передаётся сигнал пожарной сигнализации', 'fire_alarm_system_availability': 'наличие пожарной сигнализации', 'fire_compartment_areas_sq_m': 'площади пожарных отсеков, кв. м', 'fire_compartments_count': 'количество пожарных отсеков', 'fire_extinguishers_count': 'количество огнетушителей', 'fire_suppression_coverage_percent': 'доля здания, покрытая системой пожаротушения', 'fire_suppression_system_availability': 'наличие системы пожаротушения', 'fire_suppression_system_type': 'тип системы пожаротушения', 'fire_suppression_tiers_count': 'количество ярусов пожаротушения', 'fire_water_reservoir_availability': 'наличие пожарного резервуара воды', 'fire_water_reservoir_capacity_m3': 'объём пожарного резервуара, куб. м', 'functional_purpose': 'функциональное назначение объекта', 'gas_supply': 'газоснабжение', 'goods_nomenclature_group': 'номенклатурная группа товаров', 'goods_storage_method': 'способ хранения товаров', 'goods_storage_violation': 'нарушения правил хранения товаров', 'has_facade_insulation': 'есть утепление фасада', 'has_hazardous_facilities_nearby': 'рядом находятся опасные объекты', 'has_internal_insulation': 'есть внутреннее утепление', 'has_third_party_access_restriction': 'ограничен доступ посторонних лиц', 'has_unsatisfied_regulatory_prescriptions': 'есть неисполненные предписания надзорных органов', 'interfloor_overlap_material': 'материал межэтажных перекрытий', 'internal_insulation_material': 'материал внутреннего утепления', 'interrack_intertier_fire_suppression': 'межстеллажная система пожаротушения', 'last_capital_repair_year': 'год последнего капитального ремонта', 'load_bearing_walls_material': 'материал несущих стен', 'maintenance_description': 'описание технического обслуживания', 'manufacture_year': 'год производства', 'max_storage_height_m': 'максимальная высота хранения, м', 'model': 'модель оборудования или объекта', 'multi_level_mezzanine_count': 'количество многоуровневых мезонинов', 'natural_disasters_description': 'описание воздействия стихийных бедствий', 'nearby_hazardous_facility_types': 'типы опасных объектов поблизости', 'nearest_fire_station_distance_km': 'расстояние до ближайшей пожарной части, км', 'occupied_area_sq_m': 'занимаемая площадь, кв. м', 'occupied_floor': 'занимаемый этаж', 'partitions_material': 'материал перегородок', 'physical_security_availability': 'наличие физической охраны', 'power_supply': 'электроснабжение', 'power_supply_category': 'категория электроснабжения', 'primary_fire_extinguishing_means_availability': 'наличие первичных средств пожаротушения', 'primary_fire_extinguishing_means_list': 'перечень первичных средств пожаротушения', 'roofing_material': 'материал кровли', 'security_alarm_availability': 'наличие охранной сигнализации', 'security_alarm_monitoring_destination': 'куда передаётся сигнал охранной сигнализации', 'stored_goods_type': 'вид хранимых товаров', 'total_area_sq_m': 'общая площадь, кв. м', 'total_floors_count': 'общее количество этажей', 'transformer_substations': 'трансформаторные подстанции', 'transformers': 'трансформаторы', 'warehouse_clear_height': 'полезная высота склада', 'warehouse_heating_system': 'система отопления склада', 'warehouse_occupancy_percent': 'процент заполнения склада', 'was_affected_by_natural_disasters': 'объект подвергался стихийным бедствиям', 'water_network_static_pressure_atm': 'статическое давление в водопроводной сети, атм', 'water_network_static_pressure_comment': 'комментарий к давлению в водопроводной сети', 'water_supply_availability': 'наличие водоснабжения', 'water_supply_sources': 'источники водоснабжения', 'exclude_country': 'исключённая страна', 'exclude_region': 'исключённый регион', 'include_country': 'включённая страна', 'lessor_id': 'идентификатор арендодателя', 'pledgee_id': 'идентификатор залогодержателя', 'project_comment': 'комментарий по проекту', 'deposit_pub_prem_field_changed_manual': 'признак ручного изменения премии', 'req_curator_change_notify': 'уведомление о смене куратора заявки', 'rub_premium_field_changed_manual': 'признак ручного изменения премии в рублях', 'email_for_send_scans': 'адрес электронной почты для отправки сканов', 'ins_app_form': 'форма заявления на страхование', 'payer_type': 'тип плательщика', 'payer_id': 'идентификатор плательщика', 'is_filled_contract_num_policy_1c_field': 'номер договора или полиса заполнен в 1С', 'contract_count': 'количество договоров, связанных с объектом', 'contract_link_status': 'результат связи объекта с договором', 'has_address': 'есть адрес объекта', 'has_contract': 'есть связь с договором', 'has_target': 'заполнена целевая страховая сумма', 'row_source': 'источник строки датасета', 'sphere_row_id': 'номер строки датасета', 'target_status': 'результат проверки целевой переменной', 'egrn_address_candidate_count': 'количество зданий ЕГРН по ФИАС дома', 'egrn_candidate_count': 'количество подходящих зданий ЕГРН', 'egrn_is_unique_match': 'найдено ровно одно здание ЕГРН', 'address': 'адрес здания из ЕГРН', 'area_narrowed_search': 'площадь использовалась для сужения поиска', 'join_level': 'уровень соединения с ЕГРН', 'match_basis': 'основание соединения с ЕГРН', 'match_status': 'результат соединения с ЕГРН'}

column_name_translations.update({'activity_types': 'виды деятельности на объекте', 'block': 'корпус здания', 'building': 'строение', 'business_segment': 'бизнес-сегмент', 'capital_repair_year': 'год капитального ремонта', 'characteristics_version_end_date': 'дата окончания версии характеристик', 'characteristics_version_is_active': 'признак активной версии характеристик', 'characteristics_version_number': 'номер версии характеристик', 'characteristics_version_start_date': 'дата начала версии характеристик', 'compared_sphere_house': 'дом из Сферы для сравнения', 'compared_sphere_korpus': 'корпус из Сферы для сравнения', 'compared_sphere_locality': 'населённый пункт из Сферы для сравнения', 'compared_sphere_postal_code': 'почтовый индекс из Сферы для сравнения', 'compared_sphere_region': 'регион из Сферы для сравнения', 'compared_sphere_street': 'улица из Сферы для сравнения', 'compared_sphere_stroenie': 'строение из Сферы для сравнения', 'condition_count': 'количество записей с условиями страхования объекта', 'condition_currency_count': 'количество валют в условиях объекта', 'condition_max_insured_sum': 'максимальная страховая сумма в условиях объекта', 'condition_min_insured_sum': 'минимальная страховая сумма в условиях объекта', 'contract_amount_currency': 'валюта страховой суммы договора', 'contract_conclusion_date': 'дата заключения договора', 'contract_end_date': 'дата окончания договора', 'contract_insured_sum': 'общая страховая сумма договора', 'contract_premium': 'общая страховая премия договора', 'contract_sign_date': 'дата подписания договора', 'contract_start_date': 'дата начала договора', 'contract_status': 'статус договора', 'crm_macroindustry': 'макроотрасль клиента из CRM', 'crm_okved': 'ОКВЭД клиента из CRM', 'crm_segment': 'сегмент клиента из CRM', 'district': 'район', 'fias_code': 'код ФИАС адреса', 'flat': 'квартира или помещение', 'floors_count': 'количество этажей', 'house': 'номер дома', 'ins_document_type': 'тип страхового документа', 'ins_refuse': 'установлен отказ в страховании', 'insurance_product': 'страховой продукт', 'insurance_territory': 'территория страхования', 'insurance_value': 'страховая стоимость объекта', 'insurance_value_basis': 'основание определения страховой стоимости', 'insurance_value_currency': 'валюта страховой стоимости', 'insured_components': 'что входит в страхование', 'insured_sum_currency': 'валюта страховой суммы объекта', 'is_leased': 'объект находится в аренде', 'is_pledged': 'объект находится в залоге', 'latitude': 'широта', 'longitude': 'долгота', 'matched_egrn_house': 'дом найденного объекта ЕГРН', 'matched_egrn_korpus': 'корпус найденного объекта ЕГРН', 'matched_egrn_locality': 'населённый пункт найденного объекта ЕГРН', 'matched_egrn_postal_code': 'почтовый индекс найденного объекта ЕГРН', 'matched_egrn_region': 'регион найденного объекта ЕГРН', 'matched_egrn_street': 'улица найденного объекта ЕГРН', 'matched_egrn_stroenie': 'строение найденного объекта ЕГРН', 'maximum_per_occurrence_limit': 'максимальный лимит на один страховой случай', 'minimum_per_occurrence_limit': 'минимальный лимит на один страховой случай', 'object_characteristics_json': 'характеристики объекта в JSON', 'object_name': 'название объекта', 'object_type': 'тип объекта', 'occupied_area': 'занимаемая площадь', 'office': 'номер офиса', 'overlap_material': 'материал перекрытий', 'ownership_type': 'форма владения объектом', 'parsed_house': 'дом после разбора адреса', 'parsed_korpus': 'корпус после разбора адреса', 'parsed_landmark': 'ориентир после разбора адреса', 'parsed_locality': 'населённый пункт после разбора адреса', 'parsed_postal_code': 'почтовый индекс после разбора адреса', 'parsed_region': 'регион после разбора адреса', 'parsed_street': 'улица после разбора адреса', 'parsed_stroenie': 'строение после разбора адреса', 'pledged_value': 'залоговая стоимость объекта', 'policyholder_name': 'наименование страхователя', 'postal_code': 'почтовый индекс', 'previous_contract_end_date': 'дата окончания предыдущего договора', 'previous_contract_number': 'номер предыдущего договора', 'previous_contract_start_date': 'дата начала предыдущего договора', 'real_estate_objects_in_contract': 'количество объектов недвижимости в договоре', 'risk_natures': 'характеры риска', 'settlement': 'населённый пункт', 'sphere_address_for_egrn': 'адрес Сферы, использованный для поиска ЕГРН', 'street': 'улица', 'task_industry': 'отрасль из задачи оформления', 'task_object_insured_sum': 'страховая сумма объекта из связи с задачей', 'task_object_insured_sum_currency': 'валюта страховой суммы из связи с задачей', 'task_status': 'статус задачи оформления', 'task_subindustry': 'подотрасль из задачи оформления', 'task_type': 'тип задачи', 'walls_material': 'материал стен', 'address_dgis_id': 'идентификатор адреса 2ГИС', 'address_region_id': 'идентификатор региона адреса', 'characteristics_id': 'идентификатор версии характеристик объекта', 'corporate_crm_id': 'идентификатор карточки клиента в CRM', 'geo_address_id': 'идентификатор адреса объекта', 'policyholder_cdi_id': 'идентификатор страхователя в CDI', 'policyholder_id': 'идентификатор страхователя', 'previous_contract_id': 'идентификатор предыдущего договора', 'request_id': 'идентификатор заявки', 'root_contract_id': 'идентификатор первого договора в цепочке', 'task_id': 'идентификатор задачи оформления', 'task_object_link_id': 'идентификатор связи задачи и объекта', 'as_of_date': 'дата, на которую сформированы признаки', 'point__coordinates': 'координаты адреса', 'point__crs__properties__name': 'название системы координат', 'point__crs__type': 'тип системы координат', 'point__type': 'тип географической точки'})



def feature_origin(column, source):
    if column.startswith('sphere__'):
        parts = column.split('__', 2)
        if len(parts) == 3:
            return parts[1].upper(), parts[2]
    if column.startswith('egrn_raw__'):
        return 'EGRN_DATA', column.removeprefix('egrn_raw__')
    if column in exact_source_map:
        return exact_source_map[column]
    if source == 'ЕГРН':
        return 'EGRN_DATA', column.removeprefix('egrn_')
    if source == 'CDI':
        return 'DM_MOTOR.F_GET_CDI_ADDR_BY_TEXT', column
    if source == 'Технический':
        return 'расчёт в ноутбуке', column
    return None, column


def dictionary_info(column, source):
    table_name, source_column = feature_origin(column, source)
    table_comment = None
    column_comment = None

    if not dictionary_rows.empty and table_name is not None:
        table_rows = dictionary_rows.loc[
            dictionary_rows['TABLE_NAME_KEY'].eq(str(table_name).upper())
        ]
        if not table_rows.empty:
            table_comment = table_rows.iloc[0].get('TABLE_COMMENTS')
        matches = table_rows.loc[
            table_rows['COLUMN_NAME_KEY'].eq(str(source_column).lower())
        ]
        if not matches.empty:
            row = matches.iloc[0]
            table_name = row.get('TABLE_NAME', table_name)
            source_column = row.get('COLUMN_NAME', source_column)
            column_comment = row.get('COLUMN_COMMENTS')

    russian_name = explicit_russian_names.get(column)
    full_translation_key = str(source_column or column).lower()
    translation_key = full_translation_key.split('__')[-1]
    if russian_name is None:
        russian_name = column_name_translations.get(full_translation_key)
    if russian_name is None:
        russian_name = column_name_translations.get(translation_key)
    if russian_name is None and column_comment is not None and not pd.isna(column_comment):
        russian_name = str(column_comment).strip()
    if not russian_name:
        russian_name = str(source_column or column).replace('_', ' ').lower()

    return table_name, source_column, table_comment, column_comment, russian_name


# 9. Функции статистического анализа


In [ ]:
def clean_string_series(series):
    return (
        series.astype('string')
        .str.replace('\u00a0', ' ', regex=False)
        .str.strip()
        .replace('', pd.NA)
    )


def numeric_view(series):
    if pd.api.types.is_bool_dtype(series):
        return pd.Series(np.nan, index=series.index), 0.0
    if pd.api.types.is_numeric_dtype(series):
        converted = pd.to_numeric(series, errors='coerce')
    else:
        cleaned = (
            clean_string_series(series)
            .str.replace(' ', '', regex=False)
            .str.replace(',', '.', regex=False)
        )
        converted = pd.to_numeric(cleaned, errors='coerce')

    original_filled = clean_string_series(series).notna().sum()
    ratio = (
        converted.notna().sum() / original_filled
        if original_filled
        else 0.0
    )
    return converted, ratio


def datetime_view(series):
    converted = pd.to_datetime(series, errors='coerce')
    original_filled = clean_string_series(series).notna().sum()
    ratio = (
        converted.notna().sum() / original_filled
        if original_filled
        else 0.0
    )
    return converted, ratio


def infer_data_type(column, series, role):
    filled = series.dropna()
    if filled.empty:
        return 'unknown'
    if 'json' in column.lower():
        return 'json'
    if pd.api.types.is_bool_dtype(series):
        return 'boolean'

    text_values = set(
        clean_string_series(filled)
        .dropna()
        .str.lower()
        .unique()
        .tolist()
    )
    boolean_values = {
        'true', 'false', 't', 'f', 'yes', 'no', 'да', 'нет', '0', '1'
    }
    if text_values and text_values.issubset(boolean_values):
        return 'boolean'

    if date_pattern.search(column):
        _, date_ratio = datetime_view(series)
        if date_ratio >= 0.70:
            return 'datetime'

    if role != 'identifier':
        _, numeric_ratio = numeric_view(series)
        if numeric_ratio >= 0.95:
            return 'numeric'

    unique_count = filled.nunique(dropna=True)
    if unique_count <= 100 or unique_count / len(filled) <= 0.20:
        return 'category'
    return 'text'


In [ ]:
def source_available_mask(frame, source):
    if source == 'ЕГРН' and 'egrn_is_unique_match' in frame.columns:
        return pd.to_numeric(
            frame['egrn_is_unique_match'],
            errors='coerce',
        ).eq(1)
    if source == 'CDI' and 'cdi_is_unique_match' in frame.columns:
        return pd.to_numeric(
            frame['cdi_is_unique_match'],
            errors='coerce',
        ).eq(1)
    return pd.Series(True, index=frame.index)


def safe_top_values(series, feature, role, limit=10):
    if role == 'identifier' or sensitive_pattern.search(feature):
        return 'скрыто: конфиденциальное поле'

    clean = clean_string_series(series).fillna('NULL')
    counts = clean.value_counts(dropna=False).head(limit)
    total = len(clean)
    parts = []

    for value, count in counts.items():
        value_text = str(value).replace('\n', ' ').replace('|', '/')[:80]
        percent = count / total * 100 if total else 0
        parts.append(f'{value_text} [{count}, {percent:.2f}%]')

    return ' | '.join(parts)


def eta_squared(categories, target):
    pair = pd.DataFrame({'category': categories, 'target': target}).dropna()
    if len(pair) < 10 or pair['category'].nunique() < 2:
        return np.nan, len(pair)

    overall_mean = pair['target'].mean()
    total_variation = ((pair['target'] - overall_mean) ** 2).sum()
    if total_variation == 0:
        return np.nan, len(pair)

    grouped = pair.groupby('category')['target'].agg(['count', 'mean'])
    between_variation = (
        grouped['count'] * (grouped['mean'] - overall_mean) ** 2
    ).sum()
    return float(between_variation / total_variation), len(pair)


In [ ]:
def leakage_assessment(column, source, role):
    if role == 'target':
        return 'not_applicable', 'целевая переменная'
    if column in target_derived_columns:
        return 'high', 'поле напрямую связано с расчётом или наличием target'
    if role == 'identifier':
        return 'not_applicable', 'технический идентификатор'
    if role == 'service':
        return 'medium', 'служебное поле pipeline, не бизнес-признак'
    if source == 'ЕГРН' and any(
        word in column.lower()
        for word in ['update', 'actual', 'status', 'registration_date']
    ):
        return 'high', 'нужно проверить, было ли значение доступно на дату договора'
    if source in {'ЕГРН', 'CDI'}:
        return 'unknown', 'нужно подтвердить исторический срез внешнего источника'
    if date_pattern.search(column):
        return 'unknown', 'нужно проверить доступность на as_of_date'
    return 'unknown', 'требуется бизнес-проверка доступности на дату расчёта'


def preliminary_decision(role, filled_pct, unique_count, quality_flags, leakage):
    if role == 'target':
        return 'target', 'целевая переменная'
    if role == 'identifier':
        return 'exclude', 'идентификатор оставляем только для связи и контроля'
    if role == 'service':
        return 'exclude', 'служебное поле не подаём в модель'
    if 'all_missing' in quality_flags:
        return 'exclude', 'колонка полностью пустая'
    if 'constant' in quality_flags:
        return 'exclude', 'в колонке одно заполненное значение'
    if leakage == 'high':
        return 'check', 'возможна утечка данных'
    if filled_pct < 5:
        return 'check', 'заполнено меньше 5% строк'
    return 'check', 'решение принимается после бизнес-проверки и baseline-модели'


# 10. Общие метрики датасета


In [ ]:
target_numeric, target_numeric_ratio = numeric_view(df[target_column])
if target_numeric_ratio < 0.95:
    raise ValueError(
        f'Колонка {target_column} не распознана как числовая'
    )

dataset_metrics = {
    'rows': len(df),
    'columns': len(df.columns),
    'target_column': target_column,
    'target_filled': int(target_numeric.notna().sum()),
    'target_missing': int(target_numeric.isna().sum()),
    'target_zero': int(target_numeric.eq(0).sum()),
    'target_positive': int(target_numeric.gt(0).sum()),
}

if 'object_id' in df.columns:
    dataset_metrics['unique_objects'] = int(df['object_id'].nunique(dropna=True))
if 'contract_id' in df.columns:
    dataset_metrics['unique_contracts'] = int(
        df['contract_id'].nunique(dropna=True)
    )
if {'task_id', 'object_id'}.issubset(df.columns):
    duplicate_mask = df.duplicated(['task_id', 'object_id'], keep=False)
    dataset_metrics['duplicate_task_object_rows'] = int(duplicate_mask.sum())
if 'full_address' in df.columns:
    dataset_metrics['full_address_filled'] = int(
        clean_string_series(df['full_address']).notna().sum()
    )
if 'cdi_is_unique_match' in df.columns:
    dataset_metrics['cdi_unique_matches'] = int(
        pd.to_numeric(df['cdi_is_unique_match'], errors='coerce').eq(1).sum()
    )
if 'egrn_is_unique_match' in df.columns:
    dataset_metrics['egrn_unique_matches'] = int(
        pd.to_numeric(df['egrn_is_unique_match'], errors='coerce').eq(1).sum()
    )
if 'cdi_match_status' in df.columns:
    dataset_metrics['cdi_ambiguous_matches'] = int(
        df['cdi_match_status'].eq('ambiguous_house_fias').sum()
    )
if 'egrn_match_method' in df.columns:
    dataset_metrics['egrn_ambiguous_matches'] = int(
        df['egrn_match_method'].eq('ambiguous').sum()
    )
if 'as_of_date' in df.columns:
    as_of = pd.to_datetime(df['as_of_date'], errors='coerce')
    dataset_metrics['as_of_date_min'] = (
        as_of.min().date().isoformat() if as_of.notna().any() else None
    )
    dataset_metrics['as_of_date_max'] = (
        as_of.max().date().isoformat() if as_of.notna().any() else None
    )

display(
    pd.DataFrame(
        dataset_metrics.items(),
        columns=['Показатель', 'Значение'],
    )
)


# 11. Паспорт всех признаков


In [ ]:
feature_rows = []

for number, feature in enumerate(df.columns, start=1):
    raw_series = df[feature]
    if pd.api.types.is_object_dtype(raw_series) or pd.api.types.is_string_dtype(raw_series):
        series = clean_string_series(raw_series)
    else:
        series = raw_series
    source = infer_source(feature)
    role = infer_role(feature)
    data_type = infer_data_type(feature, series, role)
    available_mask = source_available_mask(df, source)

    total_rows = len(series)
    source_available_rows = int(available_mask.sum())
    filled_count = int(series.notna().sum())
    missing_count = int(series.isna().sum())
    filled_pct = filled_count / total_rows * 100 if total_rows else 0.0
    filled_among_available = int(series.loc[available_mask].notna().sum())
    filled_among_available_pct = (
        filled_among_available / source_available_rows * 100
        if source_available_rows
        else np.nan
    )
    unique_count = int(series.nunique(dropna=True))
    unique_pct = unique_count / filled_count * 100 if filled_count else 0.0

    row = {
        'record_type': 'feature',
        'metric_name': None,
        'metric_value': None,
        'feature': feature,
        'russian_name': None,
        'source': source,
        'source_table': None,
        'source_column': None,
        'source_table_comment': None,
        'source_comment': None,
        'data_type': data_type,
        'role': role,
        'total_rows': total_rows,
        'source_available_rows': source_available_rows,
        'filled_count': filled_count,
        'missing_count': missing_count,
        'filled_pct': round(filled_pct, 4),
        'filled_among_available_pct': (
            round(filled_among_available_pct, 4)
            if not pd.isna(filled_among_available_pct)
            else np.nan
        ),
        'unique_count': unique_count,
        'unique_pct': round(unique_pct, 4),
        'zero_count': None,
        'negative_count': None,
        'outlier_iqr_count': None,
        'min': None,
        'p01': None,
        'p25': None,
        'median': None,
        'mean': None,
        'p75': None,
        'p99': None,
        'max': None,
        'top_values': None,
        'target_relation_method': None,
        'target_relation_value': None,
        'target_relation_rows': None,
        'quality_flag': None,
        'available_at_prediction_time': (
            'no' if role == 'target'
            else 'not_applicable' if role in {'identifier', 'service'}
            else 'unknown'
        ),
        'leakage_risk': None,
        'leakage_reason': None,
        'preliminary_decision': None,
        'decision_reason': None,
    }

    (
        source_table,
        source_column,
        source_table_comment,
        source_comment,
        russian_name,
    ) = dictionary_info(feature, source)
    row['source_table'] = source_table
    row['source_column'] = source_column
    row['source_table_comment'] = source_table_comment
    row['source_comment'] = source_comment
    row['russian_name'] = russian_name

    quality_flags = []
    if filled_count == 0:
        quality_flags.append('all_missing')
    elif unique_count == 1:
        quality_flags.append('constant')
    if 0 < filled_pct < 5:
        quality_flags.append('coverage_lt_5pct')
    elif 5 <= filled_pct < 20:
        quality_flags.append('coverage_lt_20pct')
    if filled_count and unique_pct > 95 and role == 'feature':
        quality_flags.append('high_cardinality')

    if data_type == 'numeric':
        numeric, _ = numeric_view(series)
        valid = numeric.dropna()
        if not valid.empty:
            quantiles = valid.quantile([0.01, 0.25, 0.50, 0.75, 0.99])
            q1 = quantiles.loc[0.25]
            q3 = quantiles.loc[0.75]
            iqr = q3 - q1
            if iqr > 0:
                outlier_count = int(
                    ((valid < q1 - 1.5 * iqr) | (valid > q3 + 1.5 * iqr)).sum()
                )
            else:
                outlier_count = 0

            row.update({
                'zero_count': int(valid.eq(0).sum()),
                'negative_count': int(valid.lt(0).sum()),
                'outlier_iqr_count': outlier_count,
                'min': valid.min(),
                'p01': quantiles.loc[0.01],
                'p25': q1,
                'median': quantiles.loc[0.50],
                'mean': valid.mean(),
                'p75': q3,
                'p99': quantiles.loc[0.99],
                'max': valid.max(),
            })
            if row['negative_count']:
                quality_flags.append('contains_negative')
            if row['zero_count']:
                quality_flags.append('contains_zero')
            if outlier_count:
                quality_flags.append('iqr_outliers')

            pair = pd.DataFrame({
                'feature': numeric,
                'target': target_numeric,
            }).dropna()
            if feature != target_column and len(pair) >= 10:
                row['target_relation_method'] = 'spearman'
                row['target_relation_value'] = pair['feature'].corr(
                    pair['target'],
                    method='spearman',
                )
                row['target_relation_rows'] = len(pair)

    elif data_type == 'datetime':
        dates, _ = datetime_view(series)
        valid_dates = dates.dropna()
        if not valid_dates.empty:
            row['min'] = valid_dates.min().isoformat()
            row['max'] = valid_dates.max().isoformat()
            pair = pd.DataFrame({
                'feature': dates.map(
                    lambda value: value.toordinal() if pd.notna(value) else np.nan
                ),
                'target': target_numeric,
            }).dropna()
            if feature != target_column and len(pair) >= 10:
                row['target_relation_method'] = 'spearman_date'
                row['target_relation_value'] = pair['feature'].corr(
                    pair['target'],
                    method='spearman',
                )
                row['target_relation_rows'] = len(pair)

    elif data_type in {'category', 'boolean'}:
        row['top_values'] = safe_top_values(
            series,
            feature,
            role,
            top_values_limit,
        )
        if role == 'feature' and 2 <= unique_count <= 50:
            eta, relation_rows = eta_squared(series, target_numeric)
            row['target_relation_method'] = 'eta_squared'
            row['target_relation_value'] = eta
            row['target_relation_rows'] = relation_rows

    elif data_type in {'text', 'json'}:
        row['top_values'] = (
            'скрыто: текстовое или конфиденциальное поле'
        )

    leakage_risk, leakage_reason = leakage_assessment(
        feature,
        source,
        role,
    )
    row['leakage_risk'] = leakage_risk
    row['leakage_reason'] = leakage_reason
    row['quality_flag'] = ' | '.join(quality_flags) if quality_flags else 'ok'

    decision, decision_reason = preliminary_decision(
        role,
        filled_pct,
        unique_count,
        quality_flags,
        leakage_risk,
    )
    row['preliminary_decision'] = decision
    row['decision_reason'] = decision_reason
    feature_rows.append(row)

    if number % 50 == 0:
        print('Обработано признаков:', number)

feature_passport = pd.DataFrame(feature_rows)
print('Признаков в паспорте:', len(feature_passport))


# 12. Добавление общих метрик в тот же CSV


In [ ]:
# в csv оставляем только строки признаков
# общая сводка уже показана выше в самом ноутбуке
passport = feature_passport.copy()

value_maps = {
    'source': {
        'Сфера': 'Сфера',
        'ЕГРН': 'ЕГРН',
        'CDI': 'CDI',
        'Технический': 'расчётное поле',
    },
    'data_type': {
        'numeric': 'числовой',
        'category': 'категориальный',
        'boolean': 'логический',
        'datetime': 'дата и время',
        'text': 'текстовый',
        'json': 'JSON',
        'unknown': 'не определён',
    },
    'role': {
        'target': 'целевая переменная',
        'feature': 'кандидат в признаки',
        'identifier': 'идентификатор',
        'service': 'служебное поле',
        'split_key': 'поле разделения train/test',
    },
    'available_at_prediction_time': {
        'yes': 'да',
        'no': 'нет',
        'unknown': 'нужно проверить',
        'not_applicable': 'не применяется',
    },
    'leakage_risk': {
        'high': 'высокий',
        'medium': 'средний',
        'unknown': 'нужно проверить',
        'not_applicable': 'не применяется',
    },
    'preliminary_decision': {
        'target': 'целевая переменная',
        'check': 'проверить',
        'exclude': 'не использовать в модели',
    },
    'target_relation_method': {
        'spearman': 'корреляция Спирмена',
        'spearman_date': 'корреляция Спирмена по дате',
        'eta_squared': 'эта-квадрат',
    },
}

for column, mapping in value_maps.items():
    passport[column] = passport[column].replace(mapping)

quality_translation = {
    'all_missing': 'полностью пусто',
    'constant': 'одно значение',
    'coverage_lt_5pct': 'заполнено меньше 5%',
    'coverage_lt_20pct': 'заполнено меньше 20%',
    'high_cardinality': 'очень много уникальных значений',
    'contains_negative': 'есть отрицательные значения',
    'contains_zero': 'есть нулевые значения',
    'iqr_outliers': 'есть выбросы по IQR',
    'ok': 'явных проблем не найдено',
}


def translate_quality(value):
    if value is None or pd.isna(value):
        return None
    return ' | '.join(
        quality_translation.get(part.strip(), part.strip())
        for part in str(value).split('|')
    )


passport['quality_flag'] = passport['quality_flag'].map(translate_quality)

role_order = {
    'целевая переменная': 0,
    'кандидат в признаки': 1,
    'идентификатор': 2,
    'поле разделения train/test': 3,
    'служебное поле': 4,
}
source_order = {
    'Сфера': 0,
    'ЕГРН': 1,
    'CDI': 2,
    'расчётное поле': 3,
}
passport['_role_order'] = passport['role'].map(role_order).fillna(9)
passport['_source_order'] = passport['source'].map(source_order).fillna(9)
passport = passport.sort_values(
    ['_role_order', '_source_order', 'source_table', 'russian_name'],
    na_position='last',
).drop(columns=['_role_order', '_source_order']).reset_index(drop=True)
passport.insert(0, 'Номер', range(1, len(passport) + 1))

readable_columns = {
    'russian_name': 'Что означает колонка',
    'feature': 'Название колонки в датасете',
    'source': 'Источник',
    'source_table': 'Исходная таблица',
    'source_table_comment': 'Что хранится в таблице',
    'source_column': 'Исходная колонка',
    'source_comment': 'Описание из словаря данных',
    'role': 'Роль для модели',
    'data_type': 'Тип данных',
    'filled_pct': 'Заполнено от всех строк, %',
    'filled_among_available_pct': 'Заполнено среди доступных строк, %',
    'filled_count': 'Заполнено строк',
    'missing_count': 'Пустых строк',
    'total_rows': 'Всего строк',
    'source_available_rows': 'Строк, где источник доступен',
    'unique_count': 'Уникальных значений',
    'unique_pct': 'Уникальных от заполненных, %',
    'quality_flag': 'Что не так с данными',
    'preliminary_decision': 'Предварительное решение',
    'decision_reason': 'Почему принято такое решение',
    'available_at_prediction_time': 'Доступно на момент расчёта',
    'leakage_risk': 'Риск утечки',
    'leakage_reason': 'Почему возможна утечка',
    'top_values': 'Частые значения',
    'min': 'Минимум',
    'p01': '1-й процентиль',
    'p25': '25-й процентиль',
    'median': 'Медиана',
    'mean': 'Среднее',
    'p75': '75-й процентиль',
    'p99': '99-й процентиль',
    'max': 'Максимум',
    'zero_count': 'Количество нулей',
    'negative_count': 'Количество отрицательных значений',
    'outlier_iqr_count': 'Количество выбросов по IQR',
    'target_relation_method': 'Как измерялась связь с целью',
    'target_relation_value': 'Сила связи с целью',
    'target_relation_rows': 'Строк для расчёта связи с целью',
}

column_order = [
    'Номер',
    'russian_name',
    'feature',
    'source',
    'source_table',
    'source_table_comment',
    'source_column',
    'source_comment',
    'data_type',
    'filled_pct',
    'filled_count',
    'missing_count',
    'unique_count',
    'min',
    'median',
    'mean',
    'max',
    'zero_count',
]
passport = passport[column_order].rename(columns=readable_columns)

if len(passport) != len(df.columns):
    raise ValueError('В паспорт попали не все колонки датасета')
if passport['Название колонки в датасете'].duplicated().any():
    raise ValueError('В паспорте появились повторяющиеся признаки')

display(passport.head(30))


# 13. Контроль перед сохранением

Здесь выводятся только агрегаты. Реальные значения адресов, ИНН и идентификаторов не показываются.


In [ ]:
control = pd.DataFrame({
    'Показатель': [
        'Колонок во входном датасете',
        'Строк в итоговом CSV',
        'Числовых признаков',
        'Категориальных признаков',
        'Идентификаторов',
        'Служебных полей',
        'Полностью пустых колонок',
        'Константных колонок',
        'Признаков с высоким риском утечки',
    ],
    'Значение': [
        len(df.columns),
        len(passport),
        feature_passport['data_type'].eq('numeric').sum(),
        feature_passport['data_type'].isin(['category', 'boolean']).sum(),
        feature_passport['role'].eq('identifier').sum(),
        feature_passport['role'].eq('service').sum(),
        feature_passport['quality_flag'].str.contains('all_missing').sum(),
        feature_passport['quality_flag'].str.contains('constant').sum(),
        feature_passport['leakage_risk'].eq('high').sum(),
    ],
})
display(control)


# 14. Сохранение одного CSV


In [ ]:
results_dir.mkdir(parents=True, exist_ok=True)
passport.to_csv(
    output_file,
    index=False,
    sep=csv_separator,
    encoding=csv_encoding,
    decimal=',',
    lineterminator='\n',
)

print('Файл сохранён:', output_file)
print('Одна строка файла соответствует одной колонке датасета')
print('Строк в файле:', len(passport))


In [ ]:
engine.dispose()
khd_connection.close()
print('Подключения к Сфере и КХД закрыты')
